# Chapter 13: Pattern

> *"Each pattern describes a problem which occurs over and over again in
> our environment, and then describes the core of the solution to that
> problem, in such a way that you can use this solution a million times
> over, without ever doing it the same way twice."*  
> — **Christopher Alexander**, *A Pattern Language* (1977)

------------------------------------------------------------------------

In the earlier chapters of this book, we constructed the computational,
architectural, and type-theoretic foundations of software engineering.
We learned how physical memory is laid out, how the CPU executes machine
instructions, how programming paradigms govern state, and how algebraic
type systems establish compile-time proofs.

However, writing production systems requires more than mastering syntax
and algorithms. As codebases grow to hundreds of thousands of lines, the
primary challenge becomes **software architecture**: how components are
decoupled, how responsibilities are separated, how objects are created
without tight dependencies, and how systems remain extensible without
requiring constant rewrites.

To solve recurring structural problems, the software industry relies on
**Design Patterns**.

In 1994, Erich Gamma, Richard Helm, Ralph Johnson, and John Vlissides
(collectively known as the *Gang of Four*, or GoF) published *Design
Patterns: Elements of Reusable Object-Oriented Software*, categorising
23 canonical blueprints into three primary families:

1.  **Creational Patterns:** Blueprints for object creation mechanisms,
    isolating clients from instantiation logic.
2.  **Structural Patterns:** Blueprints for composing classes and
    objects into flexible, larger hierarchies.
3.  **Behavioural Patterns:** Blueprints for choreographing
    communication, control flow, and responsibility delegation across
    objects.

In this chapter, we will examine design patterns through the lens of
**Modern C++**. We will revisit the GoF legacy, strip away outdated
1990s idioms (such as manual raw pointer management and excessive class
inheritance), and implement them using modern constructs: RAII, smart
pointers, lambda listeners, and type-safe visitors.

Finally, we will implement an extensible, composite document and
expression engine combining **Factories**, **Composites**,
**Observers**, and **Visitors**.

### 13.1 The Modern C++ Re-Evaluation of Design Patterns

When the Gang of Four catalogued design patterns in 1994, C++ was in its
infancy. Templates were rudimentary, smart pointers did not exist in the
standard library, lambdas were unavailable, and `std::variant` was
decades away. Consequently, classical GoF implementations leaned heavily
on deep inheritance trees, raw pointers, and manual memory management.

In Modern C++, many classical patterns simplify into idiomatic language
features (see Table 13–1):

| Traditional GoF (1994 C++) | Modern C++ (C++20 / C++23) |
|:-----------------------------------|:-----------------------------------|
| Raw pointers (`T*`) and `new`/`delete` | Smart Pointers (`std::unique_ptr`, RAII) |
| Abstract Strategy interfaces | `std::function` / Generic Lambdas |
| Singleton Pattern (with locks) | Meyer's Singleton (Thread-safe static local) |
| Double-Dispatch OOP Visitor | `std::variant` and `std::visit` (Pattern Matching) |
| Heavy Abstract Factory classes | Factory functions returning `std::unique_ptr<T>` |
| Virtual method polymorphism | CRTP (Compile-time) and Type Erasure |

Table 13–1: Design Patterns Reimagined for Modern C++
{#tbl:modern-design-patterns}

Modern design patterns are not rigid recipes to be copy-pasted; they are
**mental blueprints for managing coupling and change**.

### 13.2 Creational Patterns: Factories and Safe Construction

Directly calling constructors (`new ConcreteClass()` or
`ConcreteClass{}`) tightly couples client code to specific, concrete
types. If you subsequently need to alter object lifetime policies,
substitute a mock implementation for testing, or decide the concrete
class dynamically based on configuration, every instantiation site
across the codebase must be modified.

#### 13.2.1 The Factory Method and Modern Factory Functions

A **Factory Method** encapsulates object creation behind an interface,
returning a base interface wrapped in an RAII smart pointer
(`std::unique_ptr<Interface>`).

``` cpp
#include <memory>
#include <string>
#include <iostream>
#include <format>

// Base Component Interface
class INetworkSocket {
public:
    virtual ~INetworkSocket() = default;
    virtual void connect(std::string_view host, std::uint16_t port) = 0;
};

// Concrete Implementations
class TcpSocket final : public INetworkSocket {
public:
    void connect(std::string_view host, std::uint16_t port) override {
        std::cout << std::format("Establishing TCP stream connection to {}:{}\n", host, port);
    }
};

class UdpSocket final : public INetworkSocket {
public:
    void connect(std::string_view host, std::uint16_t port) override {
        std::cout << std::format("Opening UDP datagram endpoint to {}:{}\n", host, port);
    }
};

// Modern Factory Function with Perfect Forwarding & Enums
enum class SocketProtocol { TCP, UDP };

[[nodiscard]] std::unique_ptr<INetworkSocket> create_socket(SocketProtocol proto) {
    switch (proto) {
        case SocketProtocol::TCP:
            return std::make_unique<TcpSocket>();
        case SocketProtocol::UDP:
            return std::make_unique<UdpSocket>();
    }
    throw std::invalid_argument("Unknown socket protocol.");
}
```

#### 13.2.2 The Self-Registering Extensible Factory

In large systems (such as game engines, compilers, or database drivers),
you often want new subclasses to be registered dynamically at runtime
without modifying the central factory function. We achieve this through
a **Dynamic Registry Factory** using a thread-safe registry map:

``` cpp
#include <memory>
#include <string>
#include <unordered_map>
#include <functional>
#include <stdexcept>

template <typename BaseType>
class FactoryRegistry {
public:
    using CreatorFunc = std::function<std::unique_ptr<BaseType>()>;

    static FactoryRegistry& instance() noexcept {
        // Meyer's Singleton: Guaranteed thread-safe initialisation in C++11 and later
        static FactoryRegistry registry;
        return registry;
    }

    void register_creator(const std::string& type_name, CreatorFunc creator) {
        m_creators[type_name] = std::move(creator);
    }

    [[nodiscard]] std::unique_ptr<BaseType> create(const std::string& type_name) const {
        const auto it = m_creators.find(type_name);
        if (it == m_creators.end()) {
            throw std::runtime_error("Factory: Unregistered type identifier: " + type_name);
        }
        return (it->second)(); // Invoke creator lambda
    }

private:
    FactoryRegistry() = default;
    std::unordered_map<std::string, CreatorFunc> m_creators;
};
```

### 13.3 Structural Patterns: The Composite Pattern

The **Composite Pattern** allows you to compose objects into tree
structures to represent part-whole hierarchies (see Figure 13–1).
Crucially, the pattern lets clients treat individual objects (leaf
nodes) and compositions of objects (composite branch nodes)
**uniformly**.

<figure id="fig:composite">
<img src="data:image/svg+xml;base64,PHN2ZyB2aWV3Qm94PSI3MCAyMCA4MDAgMzAwIiB3aWR0aD0iMTAwJSIgaGVpZ2h0PSJhdXRvIiB4
bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciPgogIDxkZWZzPgogICAgPHN0eWxlPgog
ICAgICA6cm9vdCB7CiAgICAgICAgLS1zdmctYmc6IHZhcigtLWJnLCAjZmZmZmZmKTsKICAgICAg
ICAtLXN2Zy1mZzogdmFyKC0tZmcsICMwMDAwMDApOwogICAgICAgIC0tc3ZnLXRpbnQtMS1mYWxs
YmFjazogI2NjY2NjYzsgLyogMjAlIGJsYWNrICsgODAlIHdoaXRlICovCiAgICAgICAgLS1zdmct
dGludC0yLWZhbGxiYWNrOiAjOTk5OTk5OyAvKiA0MCUgYmxhY2sgKyA2MCUgd2hpdGUgKi8KICAg
ICAgICAtLXN2Zy10aW50LTE6IGNvbG9yLW1peChpbiBzcmdiLCB2YXIoLS1zdmctZmcpIDIwJSwg
dmFyKC0tc3ZnLWJnKSA4MCUpOwogICAgICAgIC0tc3ZnLXRpbnQtMjogY29sb3ItbWl4KGluIHNy
Z2IsIHZhcigtLXN2Zy1mZykgNDAlLCB2YXIoLS1zdmctYmcpIDYwJSk7CiAgICAgIH0KICAgICAg
dGV4dCAgIHsgZm9udC1mYW1pbHk6ICJBcmlhbCBOYXJyb3ciLCBGcmVlU2Fucywgc2Fucy1zZXJp
ZjsgfQogICAgPC9zdHlsZT4KICAgIDxmaWx0ZXIgaWQ9ImNvbXAtc2hhZG93IiB4PSItNSUiIHk9
Ii01JSIgd2lkdGg9IjExMCUiIGhlaWdodD0iMTEwJSI+CiAgICAgIDxmZURyb3BTaGFkb3cgZHg9
IjIiIGR5PSIyIiBzdGREZXZpYXRpb249IjMiIGZsb29kLW9wYWNpdHk9IjAuMTUiLz4KICAgIDwv
ZmlsdGVyPgogICAgPG1hcmtlciBpZD0iY29tcC1hcnJvdyIgdmlld0JveD0iMCAwIDEwIDEwIiBy
ZWZYPSI2IiByZWZZPSI1IiBtYXJrZXJXaWR0aD0iNiIgbWFya2VySGVpZ2h0PSI2IiBvcmllbnQ9
ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxwYXRoIGQ9Ik0gMCAxIEwgMTAgNSBMIDAgOSB6
IiBmaWxsPSJjb250ZXh0LXN0cm9rZSIvPgogICAgPC9tYXJrZXI+CiAgPC9kZWZzPgogIDwhLS0g
QmFzZSBDb21wb25lbnQgQm94IC0tPgogIDxnIHRyYW5zZm9ybT0idHJhbnNsYXRlKDMzMCwgMzAp
IiBmaWx0ZXI9InVybCgjY29tcC1zaGFkb3cpIj4KICAgIDxyZWN0IHdpZHRoPSIyNDAiIGhlaWdo
dD0iNzAiIHJ4PSI4IiBmaWxsPSJ2YXIoLS1zdmctdGludC0xLCB2YXIoLS1zdmctdGludC0xLWZh
bGxiYWNrKSkiIHN0cm9rZT0idmFyKC0tc3ZnLWZnKSIgc3Ryb2tlLXdpZHRoPSIyIi8+CiAgICA8
dGV4dCB4PSIzNSIgeT0iMjgiIGZvbnQtc2l6ZT0iMTQiIGZvbnQtd2VpZ2h0PSJib2xkIiBmaWxs
PSJ2YXIoLS1zdmctZmcpIj4mbHQ7Jmx0O0ludGVyZmFjZSZndDsmZ3Q7IERvY05vZGU8L3RleHQ+
CiAgICA8dGV4dCB4PSIzNSIgeT0iNTIiIGZvbnQtc2l6ZT0iMTQiIGZpbGw9InZhcigtLXN2Zy1m
ZykiPisgcmVuZGVyKCkgOiB2b2lkPC90ZXh0PgogIDwvZz4KICA8IS0tIExlYWY6IFRleHQgTm9k
ZSAtLT4KICA8ZyB0cmFuc2Zvcm09InRyYW5zbGF0ZSg4MCwgMTgwKSIgZmlsdGVyPSJ1cmwoI2Nv
bXAtc2hhZG93KSI+CiAgICA8cmVjdCB3aWR0aD0iMjAwIiBoZWlnaHQ9Ijg1IiByeD0iOCIgZmls
bD0idmFyKC0tc3ZnLXRpbnQtMSwgdmFyKC0tc3ZnLXRpbnQtMS1mYWxsYmFjaykpIiBzdHJva2U9
InZhcigtLXN2Zy1mZykiIHN0cm9rZS13aWR0aD0iMS41Ii8+CiAgICA8dGV4dCB4PSIyMCIgeT0i
MjgiIGZvbnQtc2l6ZT0iMTQiIGZvbnQtd2VpZ2h0PSJib2xkIiBmaWxsPSJ2YXIoLS1zdmctZmcp
Ij5MZWFmOiBUZXh0QmxvY2s8L3RleHQ+CiAgICA8dGV4dCB4PSIyMCIgeT0iNTAiIGZvbnQtc2l6
ZT0iMTQiIGZpbGw9InZhcigtLXN2Zy1mZykiPi0gdGV4dF86IHN0cmluZzwvdGV4dD4KICAgIDx0
ZXh0IHg9IjIwIiB5PSI2OCIgZm9udC1zaXplPSIxNCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+KyBy
ZW5kZXIoKSBvdmVycmlkZTwvdGV4dD4KICA8L2c+CiAgPCEtLSBMZWFmOiBJbWFnZSBOb2RlIC0t
PgogIDxnIHRyYW5zZm9ybT0idHJhbnNsYXRlKDMyMCwgMTgwKSIgZmlsdGVyPSJ1cmwoI2NvbXAt
c2hhZG93KSI+CiAgICA8cmVjdCB3aWR0aD0iMjAwIiBoZWlnaHQ9Ijg1IiByeD0iOCIgZmlsbD0i
dmFyKC0tc3ZnLXRpbnQtMSwgdmFyKC0tc3ZnLXRpbnQtMS1mYWxsYmFjaykpIiBzdHJva2U9InZh
cigtLXN2Zy1mZykiIHN0cm9rZS13aWR0aD0iMS41Ii8+CiAgICA8dGV4dCB4PSIyMCIgeT0iMjgi
IGZvbnQtc2l6ZT0iMTQiIGZvbnQtd2VpZ2h0PSJib2xkIiBmaWxsPSJ2YXIoLS1zdmctZmcpIj5M
ZWFmOiBJbWFnZU5vZGU8L3RleHQ+CiAgICA8dGV4dCB4PSIyMCIgeT0iNTAiIGZvbnQtc2l6ZT0i
MTQiIGZpbGw9InZhcigtLXN2Zy1mZykiPi0gc3JjXzogc3RyaW5nPC90ZXh0PgogICAgPHRleHQg
eD0iMjAiIHk9IjY4IiBmb250LXNpemU9IjE0IiBmaWxsPSJ2YXIoLS1zdmctZmcpIj4rIHJlbmRl
cigpIG92ZXJyaWRlPC90ZXh0PgogIDwvZz4KICA8IS0tIENvbXBvc2l0ZTogU2VjdGlvbiAvIENv
bnRhaW5lciBOb2RlIC0tPgogIDxnIHRyYW5zZm9ybT0idHJhbnNsYXRlKDU2MCwgMTgwKSIgZmls
dGVyPSJ1cmwoI2NvbXAtc2hhZG93KSI+CiAgICA8cmVjdCB3aWR0aD0iMjgwIiBoZWlnaHQ9IjEy
MCIgcng9IjgiIGZpbGw9InZhcigtLXN2Zy10aW50LTEsIHZhcigtLXN2Zy10aW50LTEtZmFsbGJh
Y2spKSIgc3Ryb2tlPSJ2YXIoLS1zdmctZmcpIiBzdHJva2Utd2lkdGg9IjEuNSIvPgogICAgPHRl
eHQgeD0iMjAiIHk9IjI4IiBmb250LXNpemU9IjE0IiBmb250LXdlaWdodD0iYm9sZCIgZmlsbD0i
dmFyKC0tc3ZnLWZnKSI+Q29tcG9zaXRlOiBTZWN0aW9uTm9kZTwvdGV4dD4KICAgIDx0ZXh0IHg9
IjIwIiB5PSI1MCIgZm9udC1zaXplPSIxNCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+LSBjaGlsZHJl
bl86IHZlY3RvciZsdDt1bmlxdWVfcHRyJmd0OzwvdGV4dD4KICAgIDx0ZXh0IHg9IjIwIiB5PSI2
OCIgZm9udC1zaXplPSIxNCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+KyBhZGQoY2hpbGQ6IHVuaXF1
ZV9wdHIpIDogdm9pZDwvdGV4dD4KICAgIDx0ZXh0IHg9IjIwIiB5PSI4NiIgZm9udC1zaXplPSIx
NCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+KyByZW5kZXIoKSBvdmVycmlkZSB7PC90ZXh0PgogICAg
PHRleHQgeD0iMzUiIHk9IjEwMiIgZm9udC1zaXplPSIxNCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+
Zm9yKGMgOiBjaGlsZHJlbl8pIGMtJmd0O3JlbmRlcigpOyB9PC90ZXh0PgogIDwvZz4KICA8IS0t
IEluaGVyaXRhbmNlIGxpbmVzIGZyb20gSW50ZXJmYWNlIHRvIGltcGxlbWVudGF0aW9ucyAtLT4K
ICA8cGF0aCBkPSJNIDQ1MCAxMDAgTCA0NTAgMTQwIEwgMTgwIDE0MCBMIDE4MCAxNzUiIGZpbGw9
Im5vbmUiIHN0cm9rZT0idmFyKC0tc3ZnLWZnKSIgc3Ryb2tlLXdpZHRoPSIxLjUiIG1hcmtlci1l
bmQ9InVybCgjY29tcC1hcnJvdykiLz4KICA8cGF0aCBkPSJNIDQ1MCAxMDAgTCA0NTAgMTc1IiBm
aWxsPSJub25lIiBzdHJva2U9InZhcigtLXN2Zy1mZykiIHN0cm9rZS13aWR0aD0iMS41IiBtYXJr
ZXItZW5kPSJ1cmwoI2NvbXAtYXJyb3cpIi8+CiAgPHBhdGggZD0iTSA0NTAgMTAwIEwgNDUwIDE0
MCBMIDcwMCAxNDAgTCA3MDAgMTc1IiBmaWxsPSJub25lIiBzdHJva2U9InZhcigtLXN2Zy1mZyki
IHN0cm9rZS13aWR0aD0iMS41IiBtYXJrZXItZW5kPSJ1cmwoI2NvbXAtYXJyb3cpIi8+CiAgPCEt
LSBBZ2dyZWdhdGlvbiBsaW5lIGZyb20gQ29tcG9zaXRlIGJhY2sgdG8gSW50ZXJmYWNlIC0tPgog
IDxwYXRoIGQ9Ik0gODQwIDI0MCBDIDg4MCAyNDAsIDg4MCA2NSwgNTc1IDY1IiBmaWxsPSJub25l
IiBzdHJva2U9InZhcigtLXN2Zy1mZykiIHN0cm9rZS13aWR0aD0iMiIgc3Ryb2tlLWRhc2hhcnJh
eT0iNCw0IiBtYXJrZXItZW5kPSJ1cmwoI2NvbXAtYXJyb3cpIi8+CiAgPHRleHQgeD0iNzIwIiB5
PSI1NSIgZm9udC1zaXplPSIxNCIgZm9udC13ZWlnaHQ9ImJvbGQiIGZpbGw9InZhcigtLXN2Zy1m
ZykiPkNvbnRhaW5zIDAuLiogTm9kZXM8L3RleHQ+Cjwvc3ZnPg==
" alt="Figure 13–1: Composing Objects into Tree Structures" />
<figcaption>Figure 13–1: Composing Objects into Tree
Structures</figcaption>
</figure>

#### 13.3.1 Use Cases of Composite Structures

- **Document Object Models (DOM):** HTML `<div>` elements contain `<p>`,
  `<img>`, or nested `<div>` containers.
- **Abstract Syntax Trees (ASTs):** A binary arithmetic operator node
  contains two child expression nodes, each of which may be literals,
  variables, or other nested binary operators.
- **Graphical Scene Graphs:** A `Group` node contains `Mesh`, `Light`,
  or nested `Group` entities. Transforming a group automatically
  propagates transformations down through all child leaves.

### 13.4 Behavioural Patterns: Observer and Visitor

Behavioural patterns manage communication pathways and algorithms
distributed across objects.

#### 13.4.1 The Observer Pattern: Decoupled Event Broadcasting

The **Observer Pattern** defines a one-to-many dependency between
objects. When the core object (the **Subject**) undergoes a state
mutation, all registered dependents (**Observers**) are notified
automatically.

#### The Modern C++ Signal/Slot Architecture

Instead of forcing observers to inherit from a rigid `IObserver`
interface, Modern C++ uses **Type-Erased Functional Listeners**
(`std::function`) and **RAII Connection Handles**:

``` cpp
#include <iostream>
#include <vector>
#include <functional>
#include <memory>
#include <format>

class ConnectionToken {
public:
    ConnectionToken(std::function<void()> disconnect_fn)
        : m_disconnect(std::move(disconnect_fn)) {}
    
    ~ConnectionToken() {
        if (m_disconnect) {
            m_disconnect();
        }
    }

    // Move-only RAII semantics
    ConnectionToken(const ConnectionToken&) = delete;
    ConnectionToken& operator=(const ConnectionToken&) = delete;
    ConnectionToken(ConnectionToken&&) noexcept = default;
    ConnectionToken& operator=(ConnectionToken&&) noexcept = default;

private:
    std::function<void()> m_disconnect;
};

template <typename... Args>
class Signal {
public:
    using Slot = std::function<void(Args...)>;

    [[nodiscard]] std::unique_ptr<ConnectionToken> connect(Slot slot) {
        const std::size_t id = m_next_id++;
        m_slots[id] = std::move(slot);

        return std::make_unique<ConnectionToken>([this, id]() {
            m_slots.erase(id);
        });
    }

    void emit(Args... args) const {
        for (const auto& [id, slot] : m_slots) {
            slot(args...);
        }
    }

private:
    std::size_t m_next_id{0};
    std::unordered_map<std::size_t, Slot> m_slots;
};
```

*Benefits:* Observers do not need to inherit from a base class. Any
callable entity—a lambda capturing local state, a member function bound
via `std::bind`, or a free function—can act as a listener. When the
`ConnectionToken` goes out of scope, the listener automatically
unregisters itself, eliminating dangling observer bugs.

#### 13.4.2 The Visitor Pattern and The Expression Problem

One of the deepest problems in Programming Language Theory is **The
Expression Problem** (first articulated by Philip Wadler):

> *"The Expression Problem is a new name for an old problem. The goal is
> to define a datatype by cases, where one can add new cases to the
> datatype and new functions over the datatype, without recompiling
> existing code, and while retaining static type safety."*

- **Object-Oriented Programming** makes it easy to add **new data
  types** (by adding new derived subclasses), but difficult to add **new
  operations** (because adding a new virtual method forces modifying the
  base class and recompiling every subclass).
- **Functional Programming** makes it easy to add **new operations** (by
  writing a new function with pattern matching), but difficult to add
  **new data types** (because adding a new variant case forces modifying
  every existing pattern match).

The **Visitor Pattern** is the classic object-oriented solution to the
Expression Problem. It separates an algorithm from the object structure
on which it operates, allowing new operations to be defined without
altering existing class hierarchies.

#### The Mechanics of Double Dispatch

Standard C++ virtual function calls use **Single Dispatch**: the
concrete method executed depends on the runtime type of exactly one
object (`this`).

The Visitor Pattern achieves **Double Dispatch** (see Figure 13–2): the
concrete method executed depends on the runtime type of **two** distinct
objects:

1.  The runtime type of the **Element** being visited.
2.  The runtime type of the **Visitor** performing the operation.

<figure id="fig:visitor">
<img src="data:image/svg+xml;base64,PHN2ZyB2aWV3Qm94PSI1MCAyMCA4NDAgMjUwIiB3aWR0aD0iMTAwJSIgaGVpZ2h0PSIyNTAiIHht
bG5zPSJodHRwOi8vd3d3LnczLm9yZy8yMDAwL3N2ZyI+CiAgPGRlZnM+CiAgICA8c3R5bGU+CiAg
ICAgIDpyb290IHsKICAgICAgICAtLXN2Zy1iZzogdmFyKC0tYmcsICNmZmZmZmYpOwogICAgICAg
IC0tc3ZnLWZnOiB2YXIoLS1mZywgIzAwMDAwMCk7CiAgICAgICAgLS1zdmctdGludC0xLWZhbGxi
YWNrOiAjY2NjY2NjOyAvKiAyMCUgYmxhY2sgKyA4MCUgd2hpdGUgKi8KICAgICAgICAtLXN2Zy10
aW50LTItZmFsbGJhY2s6ICM5OTk5OTk7IC8qIDQwJSBibGFjayArIDYwJSB3aGl0ZSAqLwogICAg
ICAgIC0tc3ZnLXRpbnQtMTogY29sb3ItbWl4KGluIHNyZ2IsIHZhcigtLXN2Zy1mZykgMjAlLCB2
YXIoLS1zdmctYmcpIDgwJSk7CiAgICAgICAgLS1zdmctdGludC0yOiBjb2xvci1taXgoaW4gc3Jn
YiwgdmFyKC0tc3ZnLWZnKSA0MCUsIHZhcigtLXN2Zy1iZykgNjAlKTsKICAgICAgfQogICAgICB0
ZXh0ICAgeyBmb250LWZhbWlseTogIkFyaWFsIE5hcnJvdyIsIEZyZWVTYW5zLCBzYW5zLXNlcmlm
OyB9CiAgICA8L3N0eWxlPgogICAgPGZpbHRlciBpZD0idmlzLXNoYWRvdyIgeD0iLTUlIiB5PSIt
NSUiIHdpZHRoPSIxMTAlIiBoZWlnaHQ9IjExMCUiPgogICAgICA8ZmVEcm9wU2hhZG93IGR4PSIy
IiBkeT0iMiIgc3RkRGV2aWF0aW9uPSIzIiBmbG9vZC1vcGFjaXR5PSIwLjE1Ii8+CiAgICA8L2Zp
bHRlcj4KICAgIDxtYXJrZXIgaWQ9InZpcy1hcnJvdyIgdmlld0JveD0iMCAwIDEwIDEwIiByZWZY
PSI2IiByZWZZPSI1IiBtYXJrZXJXaWR0aD0iNiIgbWFya2VySGVpZ2h0PSI2IiBvcmllbnQ9ImF1
dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxwYXRoIGQ9Ik0gMCAxIEwgMTAgNSBMIDAgOSB6IiBm
aWxsPSJjb250ZXh0LXN0cm9rZSIvPgogICAgPC9tYXJrZXI+CiAgPC9kZWZzPgogIDwhLS0gQ2xp
ZW50IENvZGUgLS0+CiAgPGcgdHJhbnNmb3JtPSJ0cmFuc2xhdGUoNjUsIDMwKSIgZmlsdGVyPSJ1
cmwoI3Zpcy1zaGFkb3cpIj4KICAgIDxyZWN0IHdpZHRoPSIyMDAiIGhlaWdodD0iMjIwIiByeD0i
OCIgZmlsbD0idmFyKC0tc3ZnLXRpbnQtMSwgdmFyKC0tc3ZnLXRpbnQtMS1mYWxsYmFjaykpIiBz
dHJva2U9InZhcigtLXN2Zy1mZykiIHN0cm9rZS13aWR0aD0iMS41Ii8+CiAgICA8dGV4dCB4PSIx
NSIgeT0iMjQiIGZvbnQtc2l6ZT0iMTQiIGZvbnQtd2VpZ2h0PSJib2xkIiBmaWxsPSJ2YXIoLS1z
dmctZmcpIj4xLiBDbGllbnQgRHJpdmVyPC90ZXh0PgogICAgPHRleHQgeD0iMTUiIHk9IjY1IiBm
b250LXNpemU9IjE0IiBmaWxsPSJ2YXIoLS1zdmctZmcpIj5Eb2NOb2RlKiBub2RlID0gLi4uOzwv
dGV4dD4KICAgIDx0ZXh0IHg9IjE1IiB5PSI4NSIgZm9udC1zaXplPSIxNCIgZmlsbD0idmFyKC0t
c3ZnLWZnKSI+SHRtbFZpc2l0b3IgaHRtbDs8L3RleHQ+CiAgICA8dGV4dCB4PSIxNSIgeT0iMTI1
IiBmb250LXNpemU9IjE0IiBmb250LXdlaWdodD0iYm9sZCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+
bm9kZS0mZ3Q7YWNjZXB0KGh0bWwpOzwvdGV4dD4KICAgIDx0ZXh0IHg9IjE1IiB5PSIxNjAiIGZv
bnQtc2l6ZT0iMTQiIGZpbGw9InZhcigtLXN2Zy1mZykiPihGaXJzdCBWaXJ0dWFsIENhbGw6IDwv
dGV4dD4KICAgIDx0ZXh0IHg9IjE1IiB5PSIxNzUiIGZvbnQtc2l6ZT0iMTQiIGZpbGw9InZhcigt
LXN2Zy1mZykiPkRpc3BhdGNoZXMgb24gTm9kZSB0eXBlKTwvdGV4dD4KICA8L2c+CiAgPCEtLSBO
b2RlOiBBY2NlcHQgUGhhc2UgLS0+CiAgPHBhdGggZD0iTSAyNzAgMTI1IEwgMzM1IDEyNSIgc3Ry
b2tlPSJ2YXIoLS1zdmctZmcpIiBzdHJva2Utd2lkdGg9IjIuNSIgbWFya2VyLWVuZD0idXJsKCN2
aXMtYXJyb3cpIi8+CiAgPGcgdHJhbnNmb3JtPSJ0cmFuc2xhdGUoMzQ1LCAzMCkiIGZpbHRlcj0i
dXJsKCN2aXMtc2hhZG93KSI+CiAgICA8cmVjdCB3aWR0aD0iMjMwIiBoZWlnaHQ9IjIyMCIgcng9
IjgiIGZpbGw9InZhcigtLXN2Zy10aW50LTEsIHZhcigtLXN2Zy10aW50LTEtZmFsbGJhY2spKSIg
c3Ryb2tlPSJ2YXIoLS1zdmctZmcpIiBzdHJva2Utd2lkdGg9IjEuNSIvPgogICAgPHRleHQgeD0i
MTUiIHk9IjI0IiBmb250LXNpemU9IjE0IiBmb250LXdlaWdodD0iYm9sZCIgZmlsbD0idmFyKC0t
c3ZnLWZnKSI+Mi4gSGVhZGluZ05vZGUgKEVsZW1lbnQpPC90ZXh0PgogICAgPHRleHQgeD0iMTUi
IHk9IjY1IiBmb250LXNpemU9IjE0IiBmaWxsPSJ2YXIoLS1zdmctZmcpIj52b2lkIGFjY2VwdChJ
VmlzaXRvciZhbXA7IHYpIHs8L3RleHQ+CiAgICA8dGV4dCB4PSIyNSIgeT0iOTUiIGZvbnQtc2l6
ZT0iMTQiIGZvbnQtd2VpZ2h0PSJib2xkIiBmaWxsPSJ2YXIoLS1zdmctZmcpIj4gIHYudmlzaXQo
KnRoaXMpOzwvdGV4dD4KICAgIDx0ZXh0IHg9IjE1IiB5PSIxMjUiIGZvbnQtc2l6ZT0iMTQiIGZp
bGw9InZhcigtLXN2Zy1mZykiPn08L3RleHQ+CiAgICA8dGV4dCB4PSIxNSIgeT0iMTYwIiBmb250
LXNpemU9IjE0IiBmaWxsPSJ2YXIoLS1zdmctZmcpIj4oU2Vjb25kIFZpcnR1YWwgQ2FsbDogPC90
ZXh0PgogICAgPHRleHQgeD0iMTUiIHk9IjE3NSIgZm9udC1zaXplPSIxNCIgZmlsbD0idmFyKC0t
c3ZnLWZnKSI+RGlzcGF0Y2hlcyBvbiBWaXNpdG9yIHR5cGUsPC90ZXh0PgogICAgPHRleHQgeD0i
MTUiIHk9IjE5MCIgZm9udC1zaXplPSIxNCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+cGFzc2luZyBl
eGFjdCAqdGhpcyB0eXBlISk8L3RleHQ+CiAgPC9nPgogIDwhLS0gVmlzaXRvcjogVmlzaXQgUGhh
c2UgLS0+CiAgPHBhdGggZD0iTSA1ODAgOTUgTCA2NDUgOTUiIHN0cm9rZT0idmFyKC0tc3ZnLWZn
KSIgc3Ryb2tlLXdpZHRoPSIyLjUiIG1hcmtlci1lbmQ9InVybCgjdmlzLWFycm93KSIvPgogIDxn
IHRyYW5zZm9ybT0idHJhbnNsYXRlKDY1NSwgMzApIiBmaWx0ZXI9InVybCgjdmlzLXNoYWRvdyki
PgogICAgPHJlY3Qgd2lkdGg9IjIxMCIgaGVpZ2h0PSIyMjAiIHJ4PSI4IiBmaWxsPSJ2YXIoLS1z
dmctdGludC0xLCB2YXIoLS1zdmctdGludC0xLWZhbGxiYWNrKSkiIHN0cm9rZT0idmFyKC0tc3Zn
LWZnKSIgc3Ryb2tlLXdpZHRoPSIxLjUiLz4KICAgIDx0ZXh0IHg9IjE1IiB5PSIyNCIgZm9udC1z
aXplPSIxNCIgZm9udC13ZWlnaHQ9ImJvbGQiIGZpbGw9InZhcigtLXN2Zy1mZykiPjMuIEh0bWxW
aXNpdG9yPC90ZXh0PgogICAgPHRleHQgeD0iMTUiIHk9IjY1IiBmb250LXNpemU9IjE0IiBmaWxs
PSJ2YXIoLS1zdmctZmcpIj52aXNpdChIZWFkaW5nTm9kZSZhbXA7IGgpIHs8L3RleHQ+CiAgICA8
dGV4dCB4PSIyMCIgeT0iODUiIGZvbnQtc2l6ZT0iMTQiIGZpbGw9InZhcigtLXN2Zy1mZykiPiAg
b3V0ICs9ICImbHQ7aDEmZ3Q7IjwvdGV4dD4KICAgIDx0ZXh0IHg9IjIwIiB5PSIxMDAiIGZvbnQt
c2l6ZT0iMTQiIGZpbGw9InZhcigtLXN2Zy1mZykiPiAgICAgICAgICsgaC50ZXh0KCk8L3RleHQ+
CiAgICA8dGV4dCB4PSIyMCIgeT0iMTE1IiBmb250LXNpemU9IjE0IiBmaWxsPSJ2YXIoLS1zdmct
ZmcpIj4gICAgICAgICArICImbHQ7L2gxJmd0OyI7PC90ZXh0PgogICAgPHRleHQgeD0iMTUiIHk9
IjEzNSIgZm9udC1zaXplPSIxNCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+fTwvdGV4dD4KICAgIDx0
ZXh0IHg9IjE1IiB5PSIxNjAiIGZvbnQtc2l6ZT0iMTQiIGZpbGw9InZhcigtLXN2Zy1mZykiPlRh
cmdldCByZXNvbHZlZCB3aXRoIHplcm88L3RleHQ+CiAgICA8dGV4dCB4PSIxNSIgeT0iMTc1IiBm
b250LXNpemU9IjE0IiBmaWxsPSJ2YXIoLS1zdmctZmcpIj5ydW50aW1lIGR5bmFtaWNfY2FzdCE8
L3RleHQ+CiAgPC9nPgo8L3N2Zz4=
" alt="Figure 13–2: Double Dispatch for Visitors" />
<figcaption>Figure 13–2: Double Dispatch for Visitors</figcaption>
</figure>

### 13.5 Experimental Implementation in Modern C++: A Modular Document Processing Framework

To synthesize these patterns, we will design an extensible **Document
Processing Framework**.

Our architecture incorporates:

1.  **Creational (Factory):** A type-safe registry factory constructing
    document elements dynamically from string specifications.
2.  **Structural (Composite):** A document tree hierarchy containing
    leaf elements (Paragraphs, Headings, CodeBlocks) and composite
    elements (Sections, Document).
3.  **Behavioural (Observer):** A reactive Signal system broadcasting
    modification notifications to rendering pipelines.
4.  **Behavioural (Visitor):** Multiple independent double-dispatch
    visitors:
    - **`MarkdownExportVisitor`:** Serialises the composite document
      tree into pure Markdown syntax.
    - **`HtmlExportVisitor`:** Serialises the composite document tree
      into valid HTML5 semantic tags.
    - **`WordCountMetricsVisitor`:** Computes document statistics (word
      count, reading time).

The listings below implement this specification:

- Listing 13–1 defines the architecture header for the interfaces of
  elements and visitors.
- Listing 13–2 creates the visitors which perform the actual operations
  on the document hierarchy. Notice that we can write three completely
  independent operations across the entire composite document tree
  without modifying a single line of `DocumentTree.hpp`!
- Listing 13–3 is a demonstration driver which outputs the results from
  calling a test document with all of the visitors.

Listing 13–1: Elements and Visitor Interfaces (DocumentTree.hpp):

In [ ]:
#ifndef DOCUMENT_TREE_HPP
#define DOCUMENT_TREE_HPP

#include <string>
#include <vector>
#include <memory>
#include <functional>
#include <unordered_map>
#include <stdexcept>
#include <format>

namespace docengine {

// Forward declarations of concrete composite elements
class HeadingNode;
class ParagraphNode;
class CodeBlockNode;
class SectionNode;
class DocumentRoot;

// =========================================================================
// 1. The Abstract Visitor Interface
// =========================================================================
class IDocumentVisitor {
public:
    virtual ~IDocumentVisitor() = default;

    virtual void visit(HeadingNode& heading) = 0;
    virtual void visit(ParagraphNode& paragraph) = 0;
    virtual void visit(CodeBlockNode& code_block) = 0;
    virtual void visit(SectionNode& section) = 0;
    virtual void visit(DocumentRoot& root) = 0;
};

// =========================================================================
// 2. The Abstract Composite Node Base
// =========================================================================
class DocumentNode {
public:
    virtual ~DocumentNode() = default;

    // The double-dispatch acceptance entry point
    virtual void accept(IDocumentVisitor& visitor) = 0;

    // Optional composite child operations (default no-op for leaves)
    virtual void add(std::unique_ptr<DocumentNode> /*child*/) {
        throw std::runtime_error("Cannot add child elements to a leaf node.");
    }
};

// =========================================================================
// 3. Leaf Element Nodes
// =========================================================================

class HeadingNode final : public DocumentNode {
public:
    HeadingNode(int level, std::string text)
        : m_level(level), m_text(std::move(text)) {}

    void accept(IDocumentVisitor& visitor) override {
        visitor.visit(*this); // Dispatches on HeadingNode & Visitor type
    }

    [[nodiscard]] int level() const noexcept { return m_level; }
    [[nodiscard]] const std::string& text() const noexcept { return m_text; }

private:
    int         m_level;
    std::string m_text;
};

class ParagraphNode final : public DocumentNode {
public:
    explicit ParagraphNode(std::string text) : m_text(std::move(text)) {}

    void accept(IDocumentVisitor& visitor) override {
        visitor.visit(*this);
    }

    [[nodiscard]] const std::string& text() const noexcept { return m_text; }

private:
    std::string m_text;
};

class CodeBlockNode final : public DocumentNode {
public:
    CodeBlockNode(std::string language, std::string code)
        : m_language(std::move(language)), m_code(std::move(code)) {}

    void accept(IDocumentVisitor& visitor) override {
        visitor.visit(*this);
    }

    [[nodiscard]] const std::string& language() const noexcept { return m_language; }
    [[nodiscard]] const std::string& code() const noexcept { return m_code; }

private:
    std::string m_language;
    std::string m_code;
};

// =========================================================================
// 4. Composite Container Nodes
// =========================================================================

class SectionNode : public DocumentNode {
public:
    explicit SectionNode(std::string title) : m_title(std::move(title)) {}

    void accept(IDocumentVisitor& visitor) override {
        visitor.visit(*this);
    }

    void add(std::unique_ptr<DocumentNode> child) override {
        m_children.push_back(std::move(child));
    }

    [[nodiscard]] const std::string& title() const noexcept { return m_title; }
    [[nodiscard]] const std::vector<std::unique_ptr<DocumentNode>>& children() const noexcept {
        return m_children;
    }

private:
    std::string m_title;
    std::vector<std::unique_ptr<DocumentNode>> m_children;
};

class DocumentRoot final : public SectionNode {
public:
    explicit DocumentRoot(std::string doc_title) : SectionNode(std::move(doc_title)) {}

    void accept(IDocumentVisitor& visitor) override {
        visitor.visit(*this);
    }
};

} // namespace docengine

#endif // DOCUMENT_TREE_HPP

Listing 13–2: Concrete Visitors: Markdown, HTML, and Analytics
(Visitors.hpp):

In [ ]:
#ifndef VISITORS_HPP
#define VISITORS_HPP

//#include "DocumentTree.hpp"
#include <sstream>
#include <numeric>

namespace docengine {

// =========================================================================
// Visitor 1: Markdown Serialiser
// =========================================================================
class MarkdownExportVisitor final : public IDocumentVisitor {
public:
    void visit(HeadingNode& heading) override {
        m_ss << std::string(heading.level(), '#') << " " << heading.text() << "\n\n";
    }

    void visit(ParagraphNode& paragraph) override {
        m_ss << paragraph.text() << "\n\n";
    }

    void visit(CodeBlockNode& code_block) override {
        m_ss << "```" << code_block.language() << "\n"
             << code_block.code() << "\n```\n\n";
    }

    void visit(SectionNode& section) override {
        m_ss << "<!-- Section: " << section.title() << " -->\n";
        for (const auto& child : section.children()) {
            child->accept(*this); // Recursive composite traversal
        }
    }

    void visit(DocumentRoot& root) override {
        m_ss << "% Document: " << root.title() << "\n\n";
        for (const auto& child : root.children()) {
            child->accept(*this);
        }
    }

    [[nodiscard]] std::string get_result() const {
        return m_ss.str();
    }

private:
    std::stringstream m_ss;
};

// =========================================================================
// Visitor 2: HTML5 Semantic Serialiser
// =========================================================================
class HtmlExportVisitor final : public IDocumentVisitor {
public:
    void visit(HeadingNode& heading) override {
        m_ss << std::format("<h{0}>{1}</h{0}>\n", heading.level(), heading.text());
    }

    void visit(ParagraphNode& paragraph) override {
        m_ss << std::format("<p>{}</p>\n", paragraph.text());
    }

    void visit(CodeBlockNode& code_block) override {
        m_ss << std::format("<pre><code class=\"language-{}\">{}</code></pre>\n",
                            code_block.language(), code_block.code());
    }

    void visit(SectionNode& section) override {
        m_ss << "<section>\n";
        m_ss << std::format("  <h2>{}</h2>\n", section.title());
        for (const auto& child : section.children()) {
            child->accept(*this);
        }
        m_ss << "</section>\n";
    }

    void visit(DocumentRoot& root) override {
        m_ss << "<!DOCTYPE html>\n<html>\n<head>\n";
        m_ss << std::format("  <title>{}</title>\n", root.title());
        m_ss << "</head>\n<body>\n";
        for (const auto& child : root.children()) {
            child->accept(*this);
        }
        m_ss << "</body>\n</html>\n";
    }

    [[nodiscard]] std::string get_result() const {
        return m_ss.str();
    }

private:
    std::stringstream m_ss;
};

// =========================================================================
// Visitor 3: Analytics & Word Counter Metrics
// =========================================================================
class WordCountMetricsVisitor final : public IDocumentVisitor {
public:
    void visit(HeadingNode& heading) override {
        count_words(heading.text());
        ++m_heading_count;
    }

    void visit(ParagraphNode& paragraph) override {
        count_words(paragraph.text());
        ++m_paragraph_count;
    }

    void visit(CodeBlockNode& code_block) override {
        count_words(code_block.code());
        ++m_code_block_count;
    }

    void visit(SectionNode& section) override {
        count_words(section.title());
        for (const auto& child : section.children()) {
            child->accept(*this);
        }
    }

    void visit(DocumentRoot& root) override {
        count_words(root.title());
        for (const auto& child : root.children()) {
            child->accept(*this);
        }
    }

    [[nodiscard]] std::size_t total_words() const noexcept { return m_word_count; }
    [[nodiscard]] std::size_t heading_count() const noexcept { return m_heading_count; }
    [[nodiscard]] std::size_t paragraph_count() const noexcept { return m_paragraph_count; }
    [[nodiscard]] std::size_t code_block_count() const noexcept { return m_code_block_count; }

private:
    void count_words(std::string_view text) {
        std::stringstream ss{std::string(text)};
        std::string word;
        while (ss >> word) {
            ++m_word_count;
        }
    }

    std::size_t m_word_count{0};
    std::size_t m_heading_count{0};
    std::size_t m_paragraph_count{0};
    std::size_t m_code_block_count{0};
};

} // namespace docengine

#endif // VISITORS_HPP

Listing 13–3: Demonstration Driver: Patterns in Concert (main.cpp):

In [ ]:
//#include "DocumentTree.hpp"
//#include "Visitors.hpp"
#include <iostream>

int main() {
    using namespace docengine;

    std::cout << "========================================================\n";
    std::cout << "Composite, Visitor and Observer Patterns in Action\n";
    std::cout << "========================================================\n\n";

    // 1. Construct the Composite Document Tree
    auto document = std::make_unique<DocumentRoot>("Modern C++ Systems Architecture");

    // Add Section 1
    auto sec1 = std::make_unique<SectionNode>("Memory Subsystems");
    sec1->add(std::make_unique<HeadingNode>(2, "The Stack and Dynamic Heap"));
    sec1->add(std::make_unique<ParagraphNode>(
        "The stack provides high-speed LIFO allocation managed via the RSP hardware register, "
        "whereas the heap accommodates dynamic allocations requiring manual or RAII lifecycle management."
    ));
    sec1->add(std::make_unique<CodeBlockNode>("cpp", "std::unique_ptr<int> ptr = std::make_unique<int>(42);"));
    document->add(std::move(sec1));

    // Add Section 2
    auto sec2 = std::make_unique<SectionNode>("Behavioural Patterns");
    sec2->add(std::make_unique<HeadingNode>(2, "Double Dispatch with Visitors"));
    sec2->add(std::make_unique<ParagraphNode>(
        "The Visitor pattern decouples algorithms from data structures, resolving the Expression Problem."
    ));
    document->add(std::move(sec2));

    // -------------------------------------------------------------------------
    // 2. Execute Visitor 1: Markdown Export
    // -------------------------------------------------------------------------
    std::cout << "--- 1. MARKDOWN EXPORT VISITOR ---\n";
    MarkdownExportVisitor md_visitor;
    document->accept(md_visitor);
    std::cout << md_visitor.get_result();

    // -------------------------------------------------------------------------
    // 3. Execute Visitor 2: HTML5 Export
    // -------------------------------------------------------------------------
    std::cout << "--- 2. HTML5 EXPORT VISITOR ---\n";
    HtmlExportVisitor html_visitor;
    document->accept(html_visitor);
    std::cout << html_visitor.get_result() << "\n";

    // -------------------------------------------------------------------------
    // 4. Execute Visitor 3: Document Analytics
    // -------------------------------------------------------------------------
    std::cout << "--- 3. METRICS AND ANALYTICS VISITOR ---\n";
    WordCountMetricsVisitor metrics;
    document->accept(metrics);
    std::cout << std::format("Total Words Counted : {}\n", metrics.total_words());
    std::cout << std::format("Headings Encountered: {}\n", metrics.heading_count());
    std::cout << std::format("Paragraphs Analyzed : {}\n", metrics.paragraph_count());
    std::cout << std::format("Code Blocks Found   : {}\n\n", metrics.code_block_count());

    return 0;
}

In [ ]:
main(); // Invocation of main program for Jupyter notebook running the xeus-cpp kernel

#### 13.5.1 Program Execution Output

```
========================================================
Composite, Visitor and Observer Patterns in Action
========================================================

--- 1. MARKDOWN EXPORT VISITOR ---
% Document: Modern C++ Systems Architecture

<!-- Section: Memory Subsystems -->
## The Stack and Dynamic Heap

The stack provides high-speed LIFO allocation managed via the RSP hardware register, whereas the heap accommodates dynamic allocations requiring manual or RAII lifecycle management.

` ``cpp
std::unique_ptr<int> ptr = std::make_unique<int>(42);
` ``

<!-- Section: Behavioural Patterns -->
## Double Dispatch with Visitors

The Visitor pattern decouples algorithms from data structures, resolving the Expression Problem.


--- 2. HTML5 EXPORT VISITOR ---
<!DOCTYPE html>
<html>
<head>
  <title>Modern C++ Systems Architecture</title>
</head>
<body>
<section>
  <h2>Memory Subsystems</h2>
<h2>The Stack and Dynamic Heap</h2>
<p>The stack provides high-speed LIFO allocation managed via the RSP hardware register, whereas the heap accommodates dynamic allocations requiring manual or RAII lifecycle management.</p>
<pre><code class="language-cpp">std::unique_ptr<int> ptr = std::make_unique<int>(42);</code></pre>
</section>
<section>
  <h2>Behavioural Patterns</h2>
<h2>Double Dispatch with Visitors</h2>
<p>The Visitor pattern decouples algorithms from data structures, resolving the Expression Problem.</p>
</section>
</body>
</html>

--- 3. METRICS AND ANALYTICS VISITOR ---
Total Words Counted : 58
Headings Encountered: 2
Paragraphs Analyzed : 2
Code Blocks Found   : 1
```

### 13.6 Summary

1.  **Patterns are Architectural Blueprints:** Design patterns offer
    standard vocabularies and structural frameworks for managing change
    and complexity.
2.  **Creational Patterns (Factories):** Shield client code from
    concrete subclass instantiations, returning smart pointers
    (`std::unique_ptr<Interface>`) to maintain loose coupling.
3.  **Structural Patterns (Composites):** Enable clients to treat leaf
    objects and container branches identically across recursive
    hierarchies.
4.  **Behavioural Patterns (Observers & Visitors):**
    - The **Observer Pattern** decouples broadcast notifications using
      RAII-managed lambda slots.
    - The **Visitor Pattern** resolves the Expression Problem via Double
      Dispatch, enabling new operations to be defined across class
      hierarchies without modifying existing types.
5.  **Modern C++ Refinement:** Leverage RAII, smart pointers, lambdas,
    and type-erasure to eliminate the raw-pointer pitfalls of 1990s-era
    OOP implementations.

### 13.7 Programming Challenges

#### Exercise 13.1: The Command Pattern with Transactional Undo/Redo Stacks

**Context and Theory:**

In user-facing software, issuing an action (such as typing text,
deleting an element, or applying a graphical filter) directly from the
user interface tightly couples the UI to the underlying business domain.
Furthermore, destructive operations become irreversible.

The **Command Pattern** decouples the object invoking an operation from
the object executing it by encapsulating an action as a standalone
object.

A command object bundles three elements:

1.  A reference to the **Receiver** object whose state will be mutated.
2.  The parameters required to execute the action.
3.  The inverse parameters or state snapshots required to **undo** the
    action.

By storing executed commands inside a Last-In, First-Out (LIFO) **Undo
Stack**, an application can support arbitrary undo operations.
Symmetrically, popping from the undo stack, executing the inverse, and
pushing to a **Redo Stack** restores forward states.

$$\text{Action History}: \quad \text{UndoStack} = [C_1, C_2, C_3] \xrightarrow{\text{Undo}} \text{UndoStack} = [C_1, C_2], \quad \text{RedoStack} = [C_3]$$

If the user executes a *new* command while the redo stack is non-empty,
the redo stack must be cleared, as the timeline of actions has branched.

**Specification:**

Design a transactional text buffer using the Command Pattern:

1.  Define an abstract interface `ICommand` with pure virtual methods:
    `void execute() = 0;` and `void undo() = 0;`.
2.  Implement a `TextBuffer` receiver class storing a `std::string` with
    methods `insert(size_t pos, string text)` and
    `erase(size_t pos, size_t length)`.
3.  Implement concrete commands: `InsertCommand` and `DeleteCommand`,
    storing inverse state for undo operations.
4.  Implement a `CommandManager` maintaining an undo stack and a redo
    stack of `std::unique_ptr<ICommand>` with a configurable history
    depth limit.

``` cpp
#include <string>
#include <memory>
#include <vector>

class ICommand {
public:
    virtual ~ICommand() = default;
    virtual void execute() = 0;
    virtual void undo() = 0;
};

class TextBuffer {
    std::string m_text;
public:
    void insert(std::size_t pos, std::string_view s) { m_text.insert(pos, s); }
    void erase(std::size_t pos, std::size_t len) { m_text.erase(pos, len); }
    [[nodiscard]] const std::string& text() const noexcept { return m_text; }
};

class InsertCommand final : public ICommand {
    TextBuffer& m_buffer;
    std::size_t m_pos;
    std::string m_inserted;
public:
    InsertCommand(TextBuffer& buf, std::size_t pos, std::string text)
        : m_buffer(buf), m_pos(pos), m_inserted(std::move(text)) {}
    void execute() override { m_buffer.insert(m_pos, m_inserted); }
    void undo() override { m_buffer.erase(m_pos, m_inserted.length()); }
};
```

**Verification Tasks:**

- **Task 1 (Undo/Redo Sequence):** Start with an empty buffer. Execute
  `InsertCommand("Hello ")` followed by `InsertCommand("World")`. Verify
  the buffer reads `"Hello World"`. Execute `undo()`. Verify the buffer
  reads `"Hello "`. Execute `redo()`. Verify the buffer restores to
  `"Hello World"`.
- **Task 2 (Branching History Invalidation):** Type `"Alpha"`, then
  `"Beta"`. Undo once (buffer is `"Alpha"`). Issue a new command
  `"Gamma"`. Verify that the redo stack is cleared and that calling
  `redo()` is a no-op.
- **Task 3 (Bounded Memory History):** Configure the `CommandManager`
  with a maximum history capacity of 3. Execute 5 consecutive commands.
  Verify that the oldest 2 commands are evicted from the bottom of the
  undo stack and their memory is cleanly deallocated.

#### Exercise 13.2: Thread-Safe Reactive Observer with Weak Pointer Auto-Pruning

**Context and Theory:**

The classical Gang of Four Observer Pattern stores raw pointers to
listener interfaces (`std::vector<IObserver*>`). This design has two
major flaws:

1.  **The Lapsed Listener Problem (Memory Leak):** If an observer
    registers with a long-lived subject but is destroyed without
    explicitly calling `unregister()`, the subject retains a dangling
    pointer to deallocated memory. If the subject uses
    `std::shared_ptr<IObserver>`, the observer is kept alive
    indefinitely, causing a memory leak.
2.  **Rigid Interface Inheritance:** Observers are forced to inherit
    from an abstract base class, preventing plain lambdas, member
    functions, or free functions from acting as listeners.

In Modern C++, we resolve these problems by using **`std::weak_ptr`**
and **Type-Erased Callable Slots**.

The subject stores listeners as weak references paired with callback
functions (see Figure 13–3). During notification dispatch, the subject
locks each weak pointer:

- If the weak pointer is valid (`wp.lock() != nullptr`), the callback
  executes.
- If the weak pointer has expired (`wp.expired() == true`), the subject
  identifies that the observer has been destroyed and **prunes the dead
  slot automatically from the internal collection**.

<figure id="fig:modern-observer">
<img src="data:image/svg+xml;base64,PHN2ZyB2aWV3Qm94PSI0MCAyMCA1NjAgMTUwIiB3aWR0aD0iMTAwJSIgaGVpZ2h0PSIxNTAiIHht
bG5zPSJodHRwOi8vd3d3LnczLm9yZy8yMDAwL3N2ZyI+CiAgPGRlZnM+CiAgICA8c3R5bGU+CiAg
ICAgIDpyb290IHsKICAgICAgICAtLXN2Zy1iZzogdmFyKC0tYmcsICNmZmZmZmYpOwogICAgICAg
IC0tc3ZnLWZnOiB2YXIoLS1mZywgIzAwMDAwMCk7CiAgICAgICAgLS1zdmctdGludC0xLWZhbGxi
YWNrOiAjY2NjY2NjOyAvKiAyMCUgYmxhY2sgKyA4MCUgd2hpdGUgKi8KICAgICAgICAtLXN2Zy10
aW50LTItZmFsbGJhY2s6ICM5OTk5OTk7IC8qIDQwJSBibGFjayArIDYwJSB3aGl0ZSAqLwogICAg
ICAgIC0tc3ZnLXRpbnQtMTogY29sb3ItbWl4KGluIHNyZ2IsIHZhcigtLXN2Zy1mZykgMjAlLCB2
YXIoLS1zdmctYmcpIDgwJSk7CiAgICAgICAgLS1zdmctdGludC0yOiBjb2xvci1taXgoaW4gc3Jn
YiwgdmFyKC0tc3ZnLWZnKSA0MCUsIHZhcigtLXN2Zy1iZykgNjAlKTsKICAgICAgfQogICAgICB0
ZXh0ICAgeyBmb250LWZhbWlseTogIkFyaWFsIE5hcnJvdyIsIEZyZWVTYW5zLCBzYW5zLXNlcmlm
OyB9CiAgICA8L3N0eWxlPgogIDwvZGVmcz4KICA8cmVjdCB4PSI1MCIgeT0iMzAiIHdpZHRoPSIy
MjAiIGhlaWdodD0iMTIwIiByeD0iOCIgZmlsbD0idmFyKC0tc3ZnLXRpbnQtMSwgdmFyKC0tc3Zn
LXRpbnQtMS1mYWxsYmFjaykpIiBzdHJva2U9InZhcigtLXN2Zy1mZykiLz4KICA8dGV4dCB4PSI2
NSIgeT0iNTUiIGZvbnQtc2l6ZT0iMTQiIGZvbnQtd2VpZ2h0PSJib2xkIiBmaWxsPSJ2YXIoLS1z
dmctZmcpIj5TdWJqZWN0IChFdmVudCBQcm9kdWNlcik8L3RleHQ+CiAgPHRleHQgeD0iNjUiIHk9
IjgwIiBmb250LXNpemU9IjE0IiBmaWxsPSJ2YXIoLS1zdmctZmcpIj5zbG90czogdmVjdG9yJmx0
O1Nsb3RFbnRyeSZndDs8L3RleHQ+CiAgPHRleHQgeD0iNjUiIHk9IjEwNSIgZm9udC1zaXplPSIx
NCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+WzBdIHdlYWtfcHRyIOKGkiBBY3RpdmU8L3RleHQ+CiAg
PHRleHQgeD0iNjUiIHk9IjEyNSIgZm9udC1zaXplPSIxNCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+
WzFdIHdlYWtfcHRyIOKGkiBFWFBJUkVEITwvdGV4dD4KICA8cmVjdCB4PSIzNjAiIHk9IjMwIiB3
aWR0aD0iMjMwIiBoZWlnaHQ9IjUwIiByeD0iNiIgZmlsbD0idmFyKC0tc3ZnLXRpbnQtMSwgdmFy
KC0tc3ZnLXRpbnQtMS1mYWxsYmFjaykpIiBzdHJva2U9InZhcigtLXN2Zy1mZykiLz4KICA8dGV4
dCB4PSIzNzUiIHk9IjU1IiBmb250LXNpemU9IjE0IiBmb250LXdlaWdodD0iYm9sZCIgZmlsbD0i
dmFyKC0tc3ZnLWZnKSI+T2JzZXJ2ZXIgMSAoQWxpdmUpPC90ZXh0PgogIDx0ZXh0IHg9IjM3NSIg
eT0iNzAiIGZvbnQtc2l6ZT0iMTQiIGZpbGw9InZhcigtLXN2Zy1mZykiPnVzZV9jb3VudCAmZ3Q7
PSAxIOKGkiBEaXNwYXRjaGVkPC90ZXh0PgogIDxyZWN0IHg9IjM2MCIgeT0iMTAwIiB3aWR0aD0i
MjMwIiBoZWlnaHQ9IjUwIiByeD0iNiIgZmlsbD0idmFyKC0tc3ZnLXRpbnQtMSwgdmFyKC0tc3Zn
LXRpbnQtMS1mYWxsYmFjaykpIiBzdHJva2U9InZhcigtLXN2Zy1mZykiIHN0cm9rZS1kYXNoYXJy
YXk9IjMsMyIvPgogIDx0ZXh0IHg9IjM3NSIgeT0iMTI1IiBmb250LXNpemU9IjE0IiBmaWxsPSJ2
YXIoLS1zdmctZmcpIj5PYnNlcnZlciAyIChEZXN0cm95ZWQpPC90ZXh0PgogIDx0ZXh0IHg9IjM3
NSIgeT0iMTQwIiBmb250LXNpemU9IjE0IiBmaWxsPSJ2YXIoLS1zdmctZmcpIj5FeHBpcmVkIOKG
kiBQcnVuZWQhPC90ZXh0PgogIDxwYXRoIGQ9Ik0gMjcwIDEwMCBMIDM2MCA1NSIgc3Ryb2tlPSJ2
YXIoLS1zdmctZmcpIiBzdHJva2Utd2lkdGg9IjIiLz4KICA8cGF0aCBkPSJNIDI3MCAxMjUgTCAz
NjAgMTI1IiBzdHJva2U9InZhcigtLXN2Zy1mZykiIHN0cm9rZS13aWR0aD0iMiIgc3Ryb2tlLWRh
c2hhcnJheT0iMiwyIi8+Cjwvc3ZnPg==
" alt="Figure 13–3: Modern Observer Dispatch" />
<figcaption>Figure 13–3: Modern Observer Dispatch</figcaption>
</figure>

**Specification:**

Design a thread-safe signal dispatcher `ReactiveSignal<Args...>`:

- Maintain an internal slot registry guarded by a `std::mutex`.
- Support registering an observer tracked by a
  `std::weak_ptr<const void>`.
- Provide an `emit(Args... args)` method that iterates over slots,
  dispatches to active observers, and uses the Erase-Remove idiom to
  purge expired observers.

``` cpp
#include <functional>
#include <vector>
#include <memory>
#include <mutex>

template <typename... Args>
class ReactiveSignal {
    struct Slot {
        std::weak_ptr<const void> tracker;
        std::function<void(Args...)> callback;
    };
    mutable std::mutex m_mtx;
    std::vector<Slot> m_slots;
public:
    template <typename T>
    void connect(std::shared_ptr<T> target, std::function<void(Args...)> slot) {
        std::lock_guard<std::mutex> lock(m_mtx);
        m_slots.push_back(Slot{target, std::move(slot)});
    }
    void emit(Args... args) {
        std::lock_guard<std::mutex> lock(m_mtx);
        std::erase_if(m_slots, [](const Slot& s) { return s.tracker.expired(); });
        for (const auto& s : m_slots) s.callback(args...);
    }
    [[nodiscard]] std::size_t observer_count() const {
        std::lock_guard<std::mutex> lock(m_mtx);
        return m_slots.size();
    }
};
```

**Verification Tasks:**

- **Task 1 (Broadcast Delivery):** Register two shared pointers (`obs1`,
  `obs2`) to the signal. Emit a message. Verify that both observers
  receive the payload.
- **Task 2 (Automatic Pruning Verification):** Allow `obs2` to fall out
  of scope while `obs1` remains alive. Emit another message. Verify that
  `obs1` receives the message and `observer_count()` drops from `2` to
  `1`.
- **Task 3 (Thread-Safe Concurrent Emission):** Emit events from three
  background threads simultaneously while registering and unregistering
  observers. Verify that no data races occur using ThreadSanitizer
  (`-fsanitize=thread`).

#### Exercise 13.3: Type-Erased Modern Visitor (The Dynamic Type Matcher)

**Context and Theory:**

The classical Gang of Four Visitor pattern suffers from the **Fragile
Visitor Problem**: adding a new element subclass forces you to update
the `IVisitor` interface, adding a new `virtual void visit(NewElement&)`
method. This triggers a cascading recompilation of *every existing
visitor* in the codebase.

In Modern C++, we can resolve this coupling by building a **Type-Erased
Visitor**.

Instead of hardcoding concrete node types into an abstract visitor base
class, we maintain a registry mapping `std::type_index` identifiers to
type-erased callback lambdas:

$$\text{Registry}: \text{std::type\_index}(\text{typeid}(T)) \longrightarrow \text{std::function}\langle\text{void}(\text{const void}*)\rangle$$

When visiting an object:

1.  The dispatcher inspects the element's dynamic runtime type via
    `typeid(*element)`.
2.  It queries the internal hash map for a registered handler matching
    that `std::type_index`.
3.  If found, it casts the pointer to the concrete type and executes the
    lambda.
4.  If no handler is registered, it falls back to a default handler
    without compilation errors.

This allows developers to write visitors that handle only the subset of
elements they care about, completely decoupling nodes from visitors.

**Specification:**

Design a `DynamicTypeVisitor`:

- Implement a `register_handler<ConcreteType>(Callable fn)` method that
  maps `typeid(ConcreteType)` to a lambda performing a
  `static_cast<const ConcreteType*>`.
- Implement a `dispatch(const BaseNode& node)` method that resolves the
  type at runtime.
- Provide an `on_unhandled(Callable fn)` fallback hook.
- Demonstrate visiting a polymorphic tree without modifying any visited
  node classes.

``` cpp
#include <unordered_map>
#include <typeindex>
#include <functional>
#include <stdexcept>

class DynamicTypeVisitor {
    using RawHandler = std::function<void(const void*)>;
    std::unordered_map<std::type_index, RawHandler> m_handlers;
    std::function<void(const std::type_info&)> m_fallback;
public:
    template <typename T, typename Callable>
    void register_handler(Callable&& fn) {
        m_handlers[std::type_index(typeid(T))] = [fn = std::forward<Callable>(fn)](const void* ptr) {
            fn(*static_cast<const T*>(ptr));
        };
    }
    void on_fallback(std::function<void(const std::type_info&)> fn) { m_fallback = std::move(fn); }
    template <typename Base>
    void dispatch(const Base& obj) const {
        auto it = m_handlers.find(std::type_index(typeid(obj)));
        if (it != m_handlers.end()) (it->second)(&obj);
        else if (m_fallback) m_fallback(typeid(obj));
    }
};
```

**Verification Tasks:**

- **Task 1 (Targeted Dispatch):** Create base `Shape` and derived
  classes `Circle`, `Square`, and `Triangle`. Register handlers for
  `Circle` and `Square`. Dispatch a `Circle` and verify its registered
  lambda executes.
- **Task 2 (Fallback Handling):** Dispatch a `Triangle` (for which no
  handler was registered). Verify that the fallback handler executes,
  logging that `Triangle` was unhandled without throwing an exception.
- **Task 3 (Decoupled Node Addition):** Introduce a new derived class
  `Hexagon` in an isolated test file. Register a handler for `Hexagon`.
  Verify that it dispatches correctly without modifying or recompiling
  the base `Shape` class or any other handlers.

#### Exercise 13.4: The Fluent Builder Pattern with Compile-Time Invariant Validation

**Context and Theory:**

Constructing complex objects with numerous optional parameters often
leads to the **Telescoping Constructor Anti-Pattern**: a class declaring
dozens of constructor overloads with confusing, identically typed
parameters:

``` cpp
// Brittle Telescoping Anti-Pattern: Which boolean is what?
HttpRequest req("https://api.io", true, false, 5000, true, "GET", nullptr);
```

The **Builder Pattern** separates object construction from
representation. A **Fluent Builder** chains method calls, returning a
reference to `*this`:

``` cpp
auto req = HttpRequestBuilder()
    .set_url("https://api.io")
    .set_method(HttpMethod::POST)
    .set_timeout_ms(5000)
    .build();
```

However, traditional runtime builders can still fail at runtime if a
developer forgets to set a required field (such as the URL).

By leveraging **Type-State Programming**, we can track which fields have
been initialised in the type system itself. The `.build()` method is
enabled (via `requires` or SFINAE) **only when all required fields have
been set**, converting runtime initialization errors into **compile-time
syntax errors**.

**Specification:**

Implement an `HttpRequestBuilder`:

- Enforce that `set_url()` and `set_method()` are mandatory, while
  `set_timeout_ms()` and `add_header()` are optional.
- Implement validation inside `.build()`:
  - Throw `std::invalid_argument` if the URL is malformed or lacks an
    `http://` or `https://` prefix.
  - Throw `std::logic_error` if the method is `POST` or `PUT` but no
    `Content-Type` header was added.
- Ensure `.build()` returns an immutable, valid `HttpRequest` object
  wrapped in `std::unique_ptr`.

``` cpp
#include <string>
#include <unordered_map>
#include <stdexcept>
#include <memory>

enum class HttpMethod { GET, POST, PUT, DELETE_OP };

class HttpRequest {
    friend class HttpRequestBuilder;
    std::string m_url;
    HttpMethod m_method;
    std::unordered_map<std::string, std::string> m_headers;
    std::size_t m_timeout_ms;
    HttpRequest() = default;
public:
    [[nodiscard]] const std::string& url() const noexcept { return m_url; }
    [[nodiscard]] HttpMethod method() const noexcept { return m_method; }
};

class HttpRequestBuilder {
    HttpRequest m_req;
    bool m_has_url{false};
public:
    HttpRequestBuilder& set_url(std::string url) {
        if (!url.starts_with("http://") && !url.starts_with("https://"))
            throw std::invalid_argument("Malformed protocol in URL");
        m_req.m_url = std::move(url);
        m_has_url = true;
        return *this;
    }
    HttpRequestBuilder& set_method(HttpMethod m) { m_req.m_method = m; return *this; }
    std::unique_ptr<HttpRequest> build() {
        if (!m_has_url) throw std::logic_error("URL is required to build HttpRequest");
        return std::unique_ptr<HttpRequest>(new HttpRequest(std::move(m_req)));
    }
};
```

**Verification Tasks:**

- **Task 1 (Successful Fluent Build):** Construct a valid GET request
  using chained calls. Verify that `.url()` returns `"https://api.io"`
  and the object is constructed successfully.
- **Task 2 (Missing Invariant Rejection):** Attempt to call `.build()`
  without calling `.set_url()`. Verify that the builder throws a
  `std::logic_error`.
- **Task 3 (Protocol Formatting Validation):** Call
  `.set_url("ftp://server.org")`. Verify that an immediate
  `std::invalid_argument` exception is thrown.

#### Exercise 13.5: Composite Pattern for Hierarchical Virtual File Systems

**Context and Theory:**

Hierarchical file systems are a classic application of the **Composite
Pattern**. A file system consists of two primary entities:

1.  **Files (Leaf Nodes):** Atomic data storage units that have a size,
    but cannot contain child nodes.
2.  **Directories (Composite Nodes):** Containers that hold zero or more
    child nodes, which may be files or other nested directories.

The core strength of the Composite Pattern is **uniformity**: an
operation like calculating total storage footprint should treat a single
file and a deeply nested directory tree identically:

$$\text{Size}(\text{Node}) = \begin{cases} \text{FileSize} & \text{if Node is File} \\ \sum_{c \in \text{Children}} \text{Size}(c) & \text{if Node is Directory} \end{cases}$$

This recursive decomposition (see Figure 13–4) allows clients to
interact with complex tree hierarchies through a simple, unified
interface.

<figure id="fig:composite-pattern">
<img src="data:image/svg+xml;base64,PHN2ZyB2aWV3Qm94PSI0MCAyMCA2ODAgMTQwIiB3aWR0aD0iMTAwJSIgaGVpZ2h0PSIxNDAiIHht
bG5zPSJodHRwOi8vd3d3LnczLm9yZy8yMDAwL3N2ZyI+CiAgPGRlZnM+CiAgICA8c3R5bGU+CiAg
ICAgIDpyb290IHsKICAgICAgICAtLXN2Zy1iZzogdmFyKC0tYmcsICNmZmZmZmYpOwogICAgICAg
IC0tc3ZnLWZnOiB2YXIoLS1mZywgIzAwMDAwMCk7CiAgICAgICAgLS1zdmctdGludC0xLWZhbGxi
YWNrOiAjY2NjY2NjOyAvKiAyMCUgYmxhY2sgKyA4MCUgd2hpdGUgKi8KICAgICAgICAtLXN2Zy10
aW50LTItZmFsbGJhY2s6ICM5OTk5OTk7IC8qIDQwJSBibGFjayArIDYwJSB3aGl0ZSAqLwogICAg
ICAgIC0tc3ZnLXRpbnQtMTogY29sb3ItbWl4KGluIHNyZ2IsIHZhcigtLXN2Zy1mZykgMjAlLCB2
YXIoLS1zdmctYmcpIDgwJSk7CiAgICAgICAgLS1zdmctdGludC0yOiBjb2xvci1taXgoaW4gc3Jn
YiwgdmFyKC0tc3ZnLWZnKSA0MCUsIHZhcigtLXN2Zy1iZykgNjAlKTsKICAgICAgfQogICAgICB0
ZXh0ICAgeyBmb250LWZhbWlseTogIkFyaWFsIE5hcnJvdyIsIEZyZWVTYW5zLCBzYW5zLXNlcmlm
OyB9CiAgICA8L3N0eWxlPgogIDwvZGVmcz4KICA8cmVjdCB4PSI1MCIgeT0iMzAiIHdpZHRoPSIx
ODAiIGhlaWdodD0iNjAiIHJ4PSI2IiBmaWxsPSJ2YXIoLS1zdmctdGludC0xLCB2YXIoLS1zdmct
dGludC0xLWZhbGxiYWNrKSkiIHN0cm9rZT0idmFyKC0tc3ZnLWZnKSIvPgogIDx0ZXh0IHg9IjY1
IiB5PSI1NSIgZm9udC1zaXplPSIxNCIgZm9udC13ZWlnaHQ9ImJvbGQiIGZpbGw9InZhcigtLXN2
Zy1mZykiPlJvb3REaXIgKENvbXBvc2l0ZSk8L3RleHQ+CiAgPHRleHQgeD0iNjUiIHk9Ijc1IiBm
b250LXNpemU9IjE0IiBmaWxsPSJ2YXIoLS1zdmctZmcpIj50b3RhbF9zaXplKCkg4oaSIDY1MCBL
QjwvdGV4dD4KICA8cmVjdCB4PSIzMDAiIHk9IjMwIiB3aWR0aD0iMTgwIiBoZWlnaHQ9IjUwIiBy
eD0iNiIgZmlsbD0idmFyKC0tc3ZnLXRpbnQtMSwgdmFyKC0tc3ZnLXRpbnQtMS1mYWxsYmFjaykp
IiBzdHJva2U9InZhcigtLXN2Zy1mZykiLz4KICA8dGV4dCB4PSIzMTUiIHk9IjU1IiBmb250LXNp
emU9IjE0IiBmb250LXdlaWdodD0iYm9sZCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+ZG9jcy8gKENv
bXBvc2l0ZSk8L3RleHQ+CiAgPHRleHQgeD0iMzE1IiB5PSI3MCIgZm9udC1zaXplPSIxNCIgZmls
bD0idmFyKC0tc3ZnLWZnKSI+dG90YWxfc2l6ZSgpIOKGkiAxNTAgS0I8L3RleHQ+CiAgPHJlY3Qg
eD0iMzAwIiB5PSIxMDAiIHdpZHRoPSIxODAiIGhlaWdodD0iNTAiIHJ4PSI2IiBmaWxsPSJ2YXIo
LS1zdmctdGludC0xLCB2YXIoLS1zdmctdGludC0xLWZhbGxiYWNrKSkiIHN0cm9rZT0idmFyKC0t
c3ZnLWZnKSIvPgogIDx0ZXh0IHg9IjMxNSIgeT0iMTI1IiBmb250LXNpemU9IjE0IiBmb250LXdl
aWdodD0iYm9sZCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+a2VybmVsLmJpbiAoTGVhZik8L3RleHQ+
CiAgPHRleHQgeD0iMzE1IiB5PSIxNDAiIGZvbnQtc2l6ZT0iMTQiIGZpbGw9InZhcigtLXN2Zy1m
ZykiPnNpemUgPSA1MDAgS0I8L3RleHQ+CiAgPHJlY3QgeD0iNTUwIiB5PSIzMCIgd2lkdGg9IjE2
MCIgaGVpZ2h0PSI0MCIgcng9IjYiIGZpbGw9InZhcigtLXN2Zy10aW50LTEsIHZhcigtLXN2Zy10
aW50LTEtZmFsbGJhY2spKSIgc3Ryb2tlPSJ2YXIoLS1zdmctZmcpIi8+CiAgPHRleHQgeD0iNTY1
IiB5PSI1NSIgZm9udC1zaXplPSIxNCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+cmVhZG1lLnR4dCAo
NTAgS0IpPC90ZXh0PgogIDxyZWN0IHg9IjU1MCIgeT0iODAiIHdpZHRoPSIxNjAiIGhlaWdodD0i
NDAiIHJ4PSI2IiBmaWxsPSJ2YXIoLS1zdmctdGludC0xLCB2YXIoLS1zdmctdGludC0xLWZhbGxi
YWNrKSkiIHN0cm9rZT0idmFyKC0tc3ZnLWZnKSIvPgogIDx0ZXh0IHg9IjU2NSIgeT0iMTA1IiBm
b250LXNpemU9IjE0IiBmaWxsPSJ2YXIoLS1zdmctZmcpIj5tYW51YWwucGRmICgxMDAgS0IpPC90
ZXh0PgogIDxwYXRoIGQ9Ik0gMjMwIDYwIEwgMzAwIDU1IiBzdHJva2U9InZhcigtLXN2Zy1mZyki
IHN0cm9rZS13aWR0aD0iMiIvPgogIDxwYXRoIGQ9Ik0gMjMwIDYwIEwgMzAwIDEyNSIgc3Ryb2tl
PSJ2YXIoLS1zdmctZmcpIiBzdHJva2Utd2lkdGg9IjIiLz4KICA8cGF0aCBkPSJNIDQ4MCA1NSBM
IDU1MCA1MCIgc3Ryb2tlPSJ2YXIoLS1zdmctZmcpIiBzdHJva2Utd2lkdGg9IjIiLz4KICA8cGF0
aCBkPSJNIDQ4MCA1NSBMIDU1MCAxMDAiIHN0cm9rZT0idmFyKC0tc3ZnLWZnKSIgc3Ryb2tlLXdp
ZHRoPSIyIi8+Cjwvc3ZnPg==
" alt="Figure 13–4: Composite Pattern for Virtual Filesystem" />
<figcaption>Figure 13–4: Composite Pattern for Virtual
Filesystem</figcaption>
</figure>

**Specification:**

Design a virtual file system using the Composite Pattern:

1.  Base interface `FSNode`: declares
    `virtual std::size_t size() const = 0;` and
    `virtual std::string name() const = 0;`.
2.  Leaf class `FileNode`: stores a name and a byte size.
3.  Composite class `DirectoryNode`: stores a collection of
    `std::unique_ptr<FSNode>`.
4.  Implement `add(std::unique_ptr<FSNode>)` and
    `find(std::string_view query) -> const FSNode*`.
5.  Implement recursive formatted printing displaying tree indentation.

``` cpp
#include <string>
#include <vector>
#include <memory>
#include <numeric>

class FSNode {
public:
    virtual ~FSNode() = default;
    [[nodiscard]] virtual std::size_t size() const noexcept = 0;
    [[nodiscard]] virtual std::string_view name() const noexcept = 0;
};

class FileNode final : public FSNode {
    std::string m_name;
    std::size_t m_bytes;
public:
    FileNode(std::string n, std::size_t b) : m_name(std::move(n)), m_bytes(b) {}
    [[nodiscard]] std::size_t size() const noexcept override { return m_bytes; }
    [[nodiscard]] std::string_view name() const noexcept override { return m_name; }
};

class DirectoryNode final : public FSNode {
    std::string m_name;
    std::vector<std::unique_ptr<FSNode>> m_children;
public:
    explicit DirectoryNode(std::string n) : m_name(std::move(n)) {}
    void add(std::unique_ptr<FSNode> child) { m_children.push_back(std::move(child)); }
    [[nodiscard]] std::string_view name() const noexcept override { return m_name; }
    [[nodiscard]] std::size_t size() const noexcept override {
        return std::accumulate(m_children.begin(), m_children.end(), std::size_t{0},
            [](std::size_t acc, const auto& child) { return acc + child->size(); });
    }
};
```

**Verification Tasks:**

- **Task 1 (Recursive Size Calculation):** Build a hierarchy: `Root`
  contains `kernel.bin` (500 KB) and subfolder `docs/`, which contains
  `readme.txt` (50 KB) and `manual.pdf` (100 KB). Assert that
  `Root.size() == 650 * 1024` bytes.
- **Task 2 (Uniform Search Operation):** Search for `"manual.pdf"`
  starting at `Root`. Verify that the traversal locates the leaf node
  and returns a pointer to it.
- **Task 3 (Ownership and Clean Teardown):** Destroy `Root`. Using the
  allocation tracker from Chapter 10, verify that all child directory
  and file nodes are recursively deallocated without memory leaks.

#### Exercise 13.6: Modern Decorator Pattern via Call Wrappers and Streams

**Context and Theory:**

The **Decorator Pattern** dynamically attaches additional
responsibilities and behaviors to an object without modifying its source
code or relying on subclassing.

Subclassing introduces a static inheritance explosion: if you have a
`Logger` and want optional timestamping, file encryption, and log-level
filtering, an inheritance hierarchy requires up to $2^N$ combinatorial
subclasses (`EncryptedTimestampedLogger`, `FilteredTimestampedLogger`,
etc.).

The Decorator Pattern solves this via **Composition**. A decorator wraps
an existing component, implements the same interface, forwards calls to
the wrapped object, and adds its own custom behavior before or after
forwarding (as shown in Figure 13–5):

<figure id="fig:call-wrappers">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI1NzcuMiIgaGVp
Z2h0PSIxNzUuNiIgY2xhc3M9InN2Z2JvYiIgdmlld0JveD0iMCAtNCA1NzcuMiAxNzUuNiI+CiAg
PHN0eWxlPi5zdmdib2IgbGluZSwgLnN2Z2JvYiBwYXRoLCAuc3ZnYm9iIGNpcmNsZSwgLnN2Z2Jv
YiByZWN0LCAuc3ZnYm9iIHBvbHlnb24gewogIHN0cm9rZTogYmxhY2s7CiAgc3Ryb2tlLXdpZHRo
OiAxLjU7CiAgc3Ryb2tlLW9wYWNpdHk6IDE7CiAgZmlsbC1vcGFjaXR5OiAxOwogIHN0cm9rZS1s
aW5lY2FwOiByb3VuZDsKICBzdHJva2UtbGluZWpvaW46IG1pdGVyOwp9Cgouc3ZnYm9iIHRleHQg
ewogIHdoaXRlLXNwYWNlOiBwcmU7CiAgZmlsbDogYmxhY2s7CiAgZm9udC1mYW1pbHk6IENvdXJp
ZXIgTmV3LCBEZWphVnUgU2FucyBNb25vLCBtb25vc3BhY2U7CiAgZm9udC1zaXplOiAxM3B4Owp9
Cgouc3ZnYm9iIHJlY3QuYmFja2Ryb3AgewogIHN0cm9rZTogbm9uZTsKICBmaWxsOiB0cmFuc3Bh
cmVudDsKfQoKLnN2Z2JvYiAuYnJva2VuIHsKICBzdHJva2UtZGFzaGFycmF5OiA4Owp9Cgouc3Zn
Ym9iIC5maWxsZWQgewogIGZpbGw6IGJsYWNrOwp9Cgouc3ZnYm9iIC5iZ19maWxsZWQgewogIGZp
bGw6IHRyYW5zcGFyZW50OwogIHN0cm9rZS13aWR0aDogMTsKfQoKLnN2Z2JvYiAubm9maWxsIHsK
ICBmaWxsOiB0cmFuc3BhcmVudDsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9hcnJvdyB7CiAgbWFy
a2VyLWVuZDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9hcnJvdyB7CiAg
bWFya2VyLXN0YXJ0OiB1cmwoI2Fycm93KTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9kaWFtb25k
IHsKICBtYXJrZXItZW5kOiB1cmwoI2RpYW1vbmQpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRf
ZGlhbW9uZCB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2RpYW1vbmQpOwp9Cgouc3ZnYm9iIC5lbmRf
bWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNjaXJjbGUpOwp9Cgouc3ZnYm9iIC5z
dGFydF9tYXJrZWRfY2lyY2xlIHsKICBtYXJrZXItc3RhcnQ6IHVybCgjY2lyY2xlKTsKfQoKLnN2
Z2JvYiAuZW5kX21hcmtlZF9vcGVuX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNvcGVuX2Np
cmNsZSk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9vcGVuX2NpcmNsZSB7CiAgbWFya2VyLXN0
YXJ0OiB1cmwoI29wZW5fY2lyY2xlKTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9iaWdfb3Blbl9j
aXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjYmlnX29wZW5fY2lyY2xlKTsKfQoKLnN2Z2JvYiAu
c3RhcnRfbWFya2VkX2JpZ19vcGVuX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2JpZ19v
cGVuX2NpcmNsZSk7Cn0KCjwvc3R5bGU+CiAgPGRlZnM+CiAgICA8bWFya2VyIGlkPSJhcnJvdyIg
dmlld0JveD0iLTIgLTIgOCA4IiByZWZYPSI0IiByZWZZPSIyIiBtYXJrZXJXaWR0aD0iNyIgbWFy
a2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxwb2x5Z29u
IHBvaW50cz0iMCwwIDAsNCA0LDIgMCwwIj48L3BvbHlnb24+CiAgICA8L21hcmtlcj4KICAgIDxt
YXJrZXIgaWQ9ImRpYW1vbmQiIHZpZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIgcmVmWT0iMiIg
bWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVy
c2UiPgogICAgICA8cG9seWdvbiBwb2ludHM9IjAsMiAyLDAgNCwyIDIsNCAwLDIiPjwvcG9seWdv
bj4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iY2lyY2xlIiB2aWV3Qm94PSIwIDAgOCA4
IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBvcmll
bnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5PSI0IiByPSIy
IiBjbGFzcz0iZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0i
b3Blbl9jaXJjbGUiIHZpZXdCb3g9IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1hcmtlcldp
ZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4KICAg
ICAgPGNpcmNsZSBjeD0iNCIgY3k9IjQiIHI9IjIiIGNsYXNzPSJiZ19maWxsZWQiPjwvY2lyY2xl
PgogICAgPC9tYXJrZXI+CiAgICA8bWFya2VyIGlkPSJiaWdfb3Blbl9jaXJjbGUiIHZpZXdCb3g9
IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9
IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAgPGNpcmNsZSBjeD0iNCIgY3k9
IjQiIHI9IjMiIGNsYXNzPSJiZ19maWxsZWQiPjwvY2lyY2xlPgogICAgPC9tYXJrZXI+CiAgPC9k
ZWZzPgogIDxyZWN0IGNsYXNzPSJiYWNrZHJvcCIgeD0iMCIgeT0iLTQiIHdpZHRoPSI1NzcuMiIg
aGVpZ2h0PSIxNzUuNiI+PC9yZWN0PgogIDxsaW5lIHgxPSI1OC41IiB5MT0iMTUuNiIgeDI9IjU4
LjUiIHkyPSIzNS4xMDAwMDIiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogIDxwb2x5Z29uIHBvaW50
cz0iNTQuNjAwMDAyLDM1LjEwMDAwMiA2Mi40LDM1LjEwMDAwMiA1OC41LDQ2LjgwMDAwMyIgY2xh
c3M9ImZpbGxlZCI+PC9wb2x5Z29uPgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjE3
LjU1MDAwMSIgeT0iNTguNSIgPlRpbWVzdGFtcERlY29yYXRvcjwvdGV4dD4KICA8bGluZSB4MT0i
NTguNSIgeTE9IjYyLjQiIHgyPSI1OC41IiB5Mj0iODEuOSIgY2xhc3M9InNvbGlkIj48L2xpbmU+
CiAgPHBvbHlnb24gcG9pbnRzPSI1NC42MDAwMDIsODEuOSA2Mi40LDgxLjkgNTguNSw5My42MDAw
MDYiIGNsYXNzPSJmaWxsZWQiPjwvcG9seWdvbj4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZl
IiB4PSIxNy41NTAwMDEiIHk9IjEwNS4zIiA+RW5jcnlwdGlvbkRlY29yYXRvcjwvdGV4dD4KICA8
bGluZSB4MT0iNTguNSIgeTE9IjEwOS4yMDAwMDUiIHgyPSI1OC41IiB5Mj0iMTI4LjciIGNsYXNz
PSJzb2xpZCI+PC9saW5lPgogIDxwb2x5Z29uIHBvaW50cz0iNTQuNjAwMDAyLDEyOC43IDYyLjQs
MTI4LjcgNTguNSwxNDAuNDAwMDEiIGNsYXNzPSJmaWxsZWQiPjwvcG9seWdvbj4KICA8dGV4dCB4
bWw6c3BhY2U9InByZXNlcnZlIiB4PSIxNy41NTAwMDEiIHk9IjE1Mi4xIiA+Q29uc29sZUxvZ2dl
cjwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIxLjk1IiB5PSI1OC41IiA+
WzwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIxNjUuNzUiIHk9IjU4LjUi
ID5dPC90ZXh0PgogIDxsaW5lIHgxPSIxNzkuNDAwMDEiIHkxPSI1NC42MDAwMDIiIHgyPSIyMTAu
NiIgeTI9IjU0LjYwMDAwMiIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgPHBvbHlnb24gcG9pbnRz
PSIyMTAuNiw1MC43IDIxOC40MDAwMSw1NC42MDAwMDIgMjEwLjYsNTguNSIgY2xhc3M9ImZpbGxl
ZCI+PC9wb2x5Z29uPgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjIyOC4xNTAwMSIg
eT0iNTguNSIgPkZvcm1hdHM6PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9
IjI5OC4zNSIgeT0iNTguNSIgPiYjMzk7WzIwMjY8L3RleHQ+CiAgPGxpbmUgeDE9IjM0My4yIiB5
MT0iNTQuNjAwMDAyIiB4Mj0iMzUxIiB5Mj0iNTQuNjAwMDAyIiBjbGFzcz0ic29saWQiPjwvbGlu
ZT4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIzNTIuOTUiIHk9IjU4LjUiID4xMDwv
dGV4dD4KICA8bGluZSB4MT0iMzY2LjYiIHkxPSI1NC42MDAwMDIiIHgyPSIzNzQuNDAwMDIiIHky
PSI1NC42MDAwMDIiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogIDx0ZXh0IHhtbDpzcGFjZT0icHJl
c2VydmUiIHg9IjM3Ni4zNSIgeT0iNTguNSIgPjI4PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0i
cHJlc2VydmUiIHg9IjM5OS43NSIgeT0iNTguNSIgPjEyOjAwOjAwXTwvdGV4dD4KICA8dGV4dCB4
bWw6c3BhY2U9InByZXNlcnZlIiB4PSI0NzcuNzUiIHk9IjU4LjUiID5TeXN0ZW08L3RleHQ+CiAg
PHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNTMyLjM1MDA0IiB5PSI1OC41IiA+SW5pdCYj
Mzk7PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjEuOTUiIHk9IjEwNS4z
IiA+WzwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIxNzMuNTUiIHk9IjEw
NS4zIiA+XTwvdGV4dD4KICA8bGluZSB4MT0iMTg3LjIwMDAxIiB5MT0iMTAxLjQiIHgyPSIyMTAu
NiIgeTI9IjEwMS40IiBjbGFzcz0ic29saWQiPjwvbGluZT4KICA8cG9seWdvbiBwb2ludHM9IjIx
MC42LDk3LjUgMjE4LjQwMDAxLDEwMS40IDIxMC42LDEwNS4zIiBjbGFzcz0iZmlsbGVkIj48L3Bv
bHlnb24+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMjI4LjE1MDAxIiB5PSIxMDUu
MyIgPkVuY3J5cHRzPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjI5OC4z
NSIgeT0iMTA1LjMiID5ieXRlczo8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIg
eD0iMzUyLjk1IiB5PSIxMDUuMyIgPiYjMzk7WDkjbUtAMSEuLi4mIzM5OzwvdGV4dD4KICA8dGV4
dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIxLjk1IiB5PSIxNTIuMSIgPls8L3RleHQ+CiAgPHRl
eHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMTI2Ljc1IiB5PSIxNTIuMSIgPl08L3RleHQ+CiAg
PGxpbmUgeDE9IjE0MC40MDAwMSIgeTE9IjE0OC4yIiB4Mj0iMjEwLjYiIHkyPSIxNDguMiIgY2xh
c3M9InNvbGlkIj48L2xpbmU+CiAgPHBvbHlnb24gcG9pbnRzPSIyMTAuNiwxNDQuMyAyMTguNDAw
MDEsMTQ4LjIgMjEwLjYsMTUyLjEiIGNsYXNzPSJmaWxsZWQiPjwvcG9seWdvbj4KICA8dGV4dCB4
bWw6c3BhY2U9InByZXNlcnZlIiB4PSIyMjguMTUwMDEiIHk9IjE1Mi4xIiA+V3JpdGVzPC90ZXh0
PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjI4Mi43NSIgeT0iMTUyLjEiID5yYXc8
L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMzEzLjk1IiB5PSIxNTIuMSIg
PmJ5dGVzPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjM2MC43NSIgeT0i
MTUyLjEiID50bzwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIzODQuMTUw
MDIiIHk9IjE1Mi4xIiA+c3RhbmRhcmQ8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2
ZSIgeD0iNDU0LjM1IiB5PSIxNTIuMSIgPmNvbnNvbGU8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNl
PSJwcmVzZXJ2ZSIgeD0iNTE2Ljc1IiB5PSIxNTIuMSIgPm91dHB1dDwvdGV4dD4KICA8dGV4dCB4
bWw6c3BhY2U9InByZXNlcnZlIiB4PSIxLjk1IiB5PSIxMS43MDAwMDEiID5sb2coJiMzOTtTeXN0
ZW0gSW5pdCYjMzk7KTwvdGV4dD4KPC9zdmc+Cg==
" alt="" />
<figcaption>Figure 13–5: Client Invocation under Decorator
Pattern</figcaption>
</figure>

**Specification:**

Design a logging pipeline using the Decorator Pattern:

1.  Define the base interface `ILogger` with
    `virtual void log(std::string_view msg) = 0;`.
2.  Implement a leaf logger `ConsoleLogger` that outputs messages
    directly to `std::cout`.
3.  Implement `TimestampDecorator`: prepends the current ISO-8601
    timestamp string.
4.  Implement `EncryptionDecorator`: applies a Caesar cipher shift
    ($+3$) or XOR cipher to the message before forwarding.
5.  Demonstrate nesting decorators dynamically at runtime.

``` cpp
#include <string>
#include <string_view>
#include <memory>
#include <iostream>

class ILogger {
public:
    virtual ~ILogger() = default;
    virtual void log(std::string_view msg) = 0;
};

class ConsoleLogger final : public ILogger {
public:
    void log(std::string_view msg) override { std::cout << msg << '\n'; }
};

class LoggerDecorator : public ILogger {
protected:
    std::unique_ptr<ILogger> m_inner;
public:
    explicit LoggerDecorator(std::unique_ptr<ILogger> inner) : m_inner(std::move(inner)) {}
};

class TimestampDecorator final : public LoggerDecorator {
public:
    using LoggerDecorator::LoggerDecorator;
    void log(std::string_view msg) override {
        const std::string formatted = "[2026-10-28] " + std::string(msg);
        m_inner->log(formatted);
    }
};
```

**Verification Tasks:**

- **Task 1 (Single Decorator Output):** Wrap `ConsoleLogger` with
  `TimestampDecorator`. Log `"Boot"`. Verify that the printed string
  begins with `"[2026-10-28] Boot"`.
- **Task 2 (Chained Decorator Transformation):** Construct a pipeline:
  `EncryptionDecorator(TimestampDecorator(ConsoleLogger))`. Verify that
  the message is timestamped first, then encrypted, before reaching the
  console.
- **Task 3 (Dynamic Swapping):** Construct two pipelines sharing
  identical underlying logging components, but configured with different
  decorator layers based on a runtime debug flag.

#### Exercise 13.7: The State Pattern: Finite State Machine Protocol Decoupling

**Context and Theory:**

State-dependent entities (such as network protocol stacks, video game
character controllers, and hardware device drivers) change their
behavior based on their internal operational state.

Procedural implementations typically use a massive
`switch(m_current_state)` statement inside every single member function.
As new states and events are added, these functions grow into
unmaintainable, error-prone monoliths.

The **State Pattern** resolves this by encapsulating each state as an
independent class derived from a common interface.

The context object (e.g., `TCPConnection`) holds a pointer to the
current state object (`std::unique_ptr<ITCPState>`). When a network
event occurs:

1.  The context delegates the event to the current state object.
2.  The state object executes the behavior appropriate for that state.
3.  The state object transitions the context to a new state by swapping
    the context's internal state pointer:

$$\text{State Transitions}: \quad \text{Closed} \xrightarrow{\text{active\_open()}} \text{SynSent} \xrightarrow{\text{receive\_syn\_ack()}} \text{Established}$$

**Specification:**

Model a TCP connection lifecycle using the State Pattern:

1.  Define the state interface `ITCPState` declaring: `open()`,
    `listen()`, `receive_syn()`, and `close()`.
2.  Define concrete states: `ClosedState`, `ListenState`,
    `SynReceivedState`, and `EstablishedState`.
3.  Define the context class `TCPConnection` which delegates events to
    the active state.
4.  If an event is invalid for the current state (e.g., calling
    `receive_syn()` while in `ClosedState`), throw a `std::logic_error`.

``` cpp
#include <memory>
#include <stdexcept>
#include <string_view>

class TCPConnection;

class ITCPState {
public:
    virtual ~ITCPState() = default;
    virtual void open(TCPConnection& ctx) = 0;
    virtual void receive_syn(TCPConnection& ctx) = 0;
    virtual void close(TCPConnection& ctx) = 0;
    [[nodiscard]] virtual std::string_view name() const noexcept = 0;
};

class TCPConnection {
    std::unique_ptr<ITCPState> m_state;
public:
    explicit TCPConnection(std::unique_ptr<ITCPState> initial);
    void change_state(std::unique_ptr<ITCPState> s) { m_state = std::move(s); }
    void open() { m_state->open(*this); }
    void receive_syn() { m_state->receive_syn(*this); }
    [[nodiscard]] std::string_view state_name() const noexcept { return m_state->name(); }
};
```

**Verification Tasks:**

- **Task 1 (Valid Handshake Sequence):** Initialise `TCPConnection` in
  `ListenState`. Trigger `receive_syn()`. Verify that the connection
  transitions to `SynReceivedState`.
- **Task 2 (Illegal Transition Trapping):** Attempt to call
  `receive_syn()` while in `ClosedState`. Verify that a
  `std::logic_error` is thrown and the state remains `ClosedState`.
- **Task 3 (Lifecycle Destruction Safety):** Transition through the
  complete connection lifecycle to `EstablishedState`, then invoke
  `close()`. Verify that state objects are deallocated cleanly without
  memory leaks.

#### Exercise 13.8: The Flyweight Pattern for Particle Physics Optimisation

**Context and Theory:**

In particle simulations, graphics engines, and document rendering
pipelines, systems often instantiate millions of small objects. Storing
complete property definitions inside every individual object exhausts
physical RAM and causes severe cache thrashing.

The **Flyweight Pattern** solves this problem by dividing an object's
state into two parts (see Figure 13–6):

1.  **Intrinsic State (Shared / Context-Free):** Heavy properties that
    are identical across many instances (e.g., 3D polygon meshes,
    textures, physical mass, drag coefficients). This data is stored
    once in a shared flyweight object in memory.
2.  **Extrinsic State (Unique / Context-Dependent):** Small, mutable
    properties unique to each instance (e.g., coordinates $(x, y, z)$,
    velocity vector $(v_x, v_y, v_z)$, lifespan).

$$\begin{aligned}
\text{Memory (Naive)} &= N \times (\text{sizeof}(\text{Extrinsic}) + \text{sizeof}(\text{Intrinsic})) \\
\text{Memory (Flyweight)} &= [N \times \text{sizeof}(\text{Extrinsic})] + [M \times \text{sizeof}(\text{Intrinsic})] \quad \text{where } M \ll N
\end{aligned}$$

For $N = 1,000,000$ particles of $M = 4$ types, Flyweight reduces memory
consumption from hundreds of megabytes to a fraction of that footprint.

<figure id="fig:flyweight-pattern">
<img src="data:image/svg+xml;base64,PHN2ZyB2aWV3Qm94PSI0MCAyMCA2NTAgMTQwIiB3aWR0aD0iMTAwJSIgaGVpZ2h0PSIxNDAiIHht
bG5zPSJodHRwOi8vd3d3LnczLm9yZy8yMDAwL3N2ZyI+CiAgPGRlZnM+CiAgICA8c3R5bGU+CiAg
ICAgIDpyb290IHsKICAgICAgICAtLXN2Zy1iZzogdmFyKC0tYmcsICNmZmZmZmYpOwogICAgICAg
IC0tc3ZnLWZnOiB2YXIoLS1mZywgIzAwMDAwMCk7CiAgICAgICAgLS1zdmctdGludC0xLWZhbGxi
YWNrOiAjY2NjY2NjOyAvKiAyMCUgYmxhY2sgKyA4MCUgd2hpdGUgKi8KICAgICAgICAtLXN2Zy10
aW50LTItZmFsbGJhY2s6ICM5OTk5OTk7IC8qIDQwJSBibGFjayArIDYwJSB3aGl0ZSAqLwogICAg
ICAgIC0tc3ZnLXRpbnQtMTogY29sb3ItbWl4KGluIHNyZ2IsIHZhcigtLXN2Zy1mZykgMjAlLCB2
YXIoLS1zdmctYmcpIDgwJSk7CiAgICAgICAgLS1zdmctdGludC0yOiBjb2xvci1taXgoaW4gc3Jn
YiwgdmFyKC0tc3ZnLWZnKSA0MCUsIHZhcigtLXN2Zy1iZykgNjAlKTsKICAgICAgfQogICAgICB0
ZXh0ICAgeyBmb250LWZhbWlseTogIkFyaWFsIE5hcnJvdyIsIEZyZWVTYW5zLCBzYW5zLXNlcmlm
OyB9CiAgICA8L3N0eWxlPgogIDwvZGVmcz4KICA8cmVjdCB4PSI1MCIgeT0iMzAiIHdpZHRoPSIy
MzAiIGhlaWdodD0iMTIwIiByeD0iOCIgZmlsbD0idmFyKC0tc3ZnLXRpbnQtMSwgdmFyKC0tc3Zn
LXRpbnQtMS1mYWxsYmFjaykpIiBzdHJva2U9InZhcigtLXN2Zy1mZykiLz4KICA8dGV4dCB4PSI2
NSIgeT0iNTUiIGZvbnQtc2l6ZT0iMTQiIGZvbnQtd2VpZ2h0PSJib2xkIiBmaWxsPSJ2YXIoLS1z
dmctZmcpIj5QYXJ0aWNsZSBBcnJheSAoMSwwMDAsMDAwKTwvdGV4dD4KICA8dGV4dCB4PSI2NSIg
eT0iODAiIGZvbnQtc2l6ZT0iMTQiIGZpbGw9InZhcigtLXN2Zy1mZykiPngsIHksIHogICAgICAg
OiAxMiBCeXRlczwvdGV4dD4KICA8dGV4dCB4PSI2NSIgeT0iOTgiIGZvbnQtc2l6ZT0iMTQiIGZp
bGw9InZhcigtLXN2Zy1mZykiPnZ4LCB2eSwgdnogICAgOiAxMiBCeXRlczwvdGV4dD4KICA8dGV4
dCB4PSI2NSIgeT0iMTE2IiBmb250LXNpemU9IjE0IiBmaWxsPSJ2YXIoLS1zdmctZmcpIj5mbHl3
ZWlnaHRfcHRyIDogIDggQnl0ZXM8L3RleHQ+CiAgPHRleHQgeD0iNjUiIHk9IjEzNiIgZm9udC1z
aXplPSIxNCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+VG90YWw6IDMyIEJ5dGVzIHBlciBQYXJ0aWNs
ZTwvdGV4dD4KICA8cmVjdCB4PSIzNjAiIHk9IjMwIiB3aWR0aD0iMjcwIiBoZWlnaHQ9IjEyMCIg
cng9IjgiIGZpbGw9InZhcigtLXN2Zy10aW50LTEsIHZhcigtLXN2Zy10aW50LTEtZmFsbGJhY2sp
KSIgc3Ryb2tlPSJ2YXIoLS1zdmctZmcpIi8+CiAgPHRleHQgeD0iMzc1IiB5PSI1NSIgZm9udC1z
aXplPSIxNCIgZm9udC13ZWlnaHQ9ImJvbGQiIGZpbGw9InZhcigtLXN2Zy1mZykiPlNoYXJlZCBG
bHl3ZWlnaHRzIChPbmx5IDIhKTwvdGV4dD4KICA8dGV4dCB4PSIzNzUiIHk9IjgwIiBmb250LXNp
emU9IjE0IiBmaWxsPSJ2YXIoLS1zdmctZmcpIj5bMF0gU21va2U6IDI1NktCIFRleHR1cmUsIE1h
c3M8L3RleHQ+CiAgPHRleHQgeD0iMzc1IiB5PSIxMDUiIGZvbnQtc2l6ZT0iMTQiIGZpbGw9InZh
cigtLXN2Zy1mZykiPlsxXSBTcGFyazogMTI4S0IgTWVzaCwgQ29sb3I8L3RleHQ+CiAgPHRleHQg
eD0iMzc1IiB5PSIxMzYiIGZvbnQtc2l6ZT0iMTQiIGZpbGw9InZhcigtLXN2Zy1mZykiPkFsbG9j
YXRlZCBPTkNFIGluIEZseXdlaWdodEZhY3Rvcnk8L3RleHQ+CiAgPHBhdGggZD0iTSAyODAgMTE2
IEwgMzYwIDgwIiBzdHJva2U9InZhcigtLXN2Zy1mZykiIHN0cm9rZS13aWR0aD0iMiIgbWFya2Vy
LWVuZD0idXJsKCNwcm9vZi1hcnJvdykiLz4KPC9zdmc+
" alt="Figure 13–6: Flyweight Pattern (Object Division)" />
<figcaption>Figure 13–6: Flyweight Pattern (Object
Division)</figcaption>
</figure>

**Specification:**

Design a particle physics memory subsystem:

1.  Define intrinsic struct `ParticleModel`: contains
    `std::array<std::uint8_t, 1024> texture_blob` and
    `double drag_coefficient`.
2.  Implement a `FlyweightFactory` that pools and dispenses
    `std::shared_ptr<const ParticleModel>` based on a string identifier
    (e.g., `"Smoke"`, `"Spark"`).
3.  Define extrinsic struct `Particle`: contains coordinates
    `double x, y, z`, velocities `double vx, vy, vz`, and a non-owning
    pointer `const ParticleModel* model`.
4.  Instantiate 500,000 `"Smoke"` particles and 500,000 `"Spark"`
    particles. Compare the memory footprint against a naive design that
    embeds the texture blob directly inside each particle.

``` cpp
#include <string>
#include <unordered_map>
#include <memory>
#include <array>

struct ParticleModel {
    std::string name;
    std::array<std::uint8_t, 1024> texture_blob;
    double drag;
};

class ParticleFactory {
    std::unordered_map<std::string, std::shared_ptr<const ParticleModel>> m_pool;
public:
    std::shared_ptr<const ParticleModel> get(const std::string& name) {
        auto it = m_pool.find(name);
        if (it != m_pool.end()) return it->second;
        auto model = std::make_shared<ParticleModel>(name, std::array<std::uint8_t, 1024>{}, 0.05);
        m_pool[name] = model;
        return model;
    }
};

struct Particle {
    double x, y, z;
    double vx, vy, vz;
    const ParticleModel* model; // Non-owning pointer to shared flyweight
};
```

**Verification Tasks:**

- **Task 1 (Factory Deduplication):** Request `"Smoke"` from the factory
  five times. Verify that all five calls return pointers to the exact
  same memory address.
- **Task 2 (Memory Footprint Reduction):** Allocate 1,000,000 particles
  using the Flyweight pattern. Verify that the total memory allocated is
  approximately $\approx 56\text{ MB}$, compared to $>1\text{ GB}$ for
  the naive architecture without flyweights.
- **Task 3 (Update Loop Consistency):** Run a simulation step advancing
  the positions of all 1,000,000 particles based on velocity and their
  shared flyweight drag coefficient. Verify numerical consistency.

#### Exercise 13.9: Benchmark: Double-Dispatch OOP Visitor vs Functional `std::visit`

**Context and Theory:**

In Chapter 12 and Chapter 13, we explored two distinct solutions to the
Expression Problem:

1.  **Classical OOP Visitor:** Implements double dispatch using two
    virtual function calls (`accept(IVisitor&)` followed by
    `visit(ConcreteNode&)`). Because neither call target can be
    determined at compile time, the compiler cannot inline the
    operations, incurring two `vtable` lookups per node.
2.  **Functional Visitor (`std::variant` + `std::visit`):** Represents
    nodes as closed sum types. The compiler translates `std::visit` into
    a single direct jump table or inline conditional branch tree,
    allowing the visitor lambdas to be inlined directly.

In tight execution loops traversing large AST trees (e.g., compilers
evaluating millions of expressions), this architectural difference can
have a major impact on runtime performance.

**Specification:**

Build an arithmetic expression tree benchmark comparing both styles:

1.  **OOP Tree:** `ExprNode` base class with derived `LiteralNode` and
    `AddNode`. Implement an `EvaluatorVisitor` using double dispatch.
2.  **Variant Tree:** `using ExprVariant = std::variant<Literal, Add>;`
    evaluated via `std::visit`.
3.  Construct an identical, deeply nested binary tree of 500,000
    additions ($1 + 1 + 1 + \dots$).
4.  Benchmark the time taken to evaluate the complete tree across 100
    iterations for both architectures.

``` cpp
#include <variant>
#include <memory>

struct Literal;
struct Add;
using VariantNode = std::variant<Literal, Add>;

struct Literal { double value; };
struct Add {
    std::shared_ptr<VariantNode> lhs;
    std::shared_ptr<VariantNode> rhs;
};

[[nodiscard]] double evaluate_variant(const VariantNode& node) {
    return std::visit([](const auto& n) -> double {
        using T = std::decay_t<decltype(n)>;
        if constexpr (std::is_same_v<T, Literal>) return n.value;
        if constexpr (std::is_same_v<T, Add>) 
            return evaluate_variant(*n.lhs) + evaluate_variant(*n.rhs);
    }, node);
}
```

**Verification Tasks:**

- **Task 1 (Mathematical Parity):** Evaluate an identical tree of
  500,000 additions across both implementations. Verify that both
  evaluate to $500,000.0$.
- **Task 2 (Execution Speed Comparison):** Benchmark both across 100
  runs. Verify that the `std::variant` implementation runs substantially
  faster (typically $2\times$ to $4\times$ faster) than the OOP
  double-dispatch visitor.
- **Task 3 (Call-Stack Depth Inspection):** Profile both implementations
  using a debugger. Observe that the OOP visitor pushes two stack frames
  per tree level (`accept` and `visit`), whereas the variant evaluator
  can inline recursive calls more aggressively.

#### Exercise 13.10: Self-Registering Factory with Static Reflection Simulation

**Context and Theory:**

Standard factory implementations require a centralised `switch`
statement or enumeration listing every concrete derived class. This
creates an architectural bottleneck: every time a developer adds a new
derived class, they must modify the central factory header and recompile
all dependent modules, violating the **Open-Closed Principle**.

A **Self-Registering Factory** allows new subclasses to register their
creation functions dynamically **without modifying the central
factory**.

In C++, this is achieved using **Static Initialization**:

1.  The central factory exposes a registration method:
    `static void register_type(string name, CreatorFunc creator);`
2.  Each derived class file declares an anonymous namespace containing a
    dummy static variable whose initialization invokes
    `register_type()`.
3.  When the operating system loads the binary or dynamic library into
    memory, static initializers run before `main()`, registering all
    types into the factory automatically.

$$\text{Global Startup}: \quad \text{Static Init} \longrightarrow \text{Factory::register}(\text{"Audio"}, \text{\&create\_audio}) \longrightarrow \text{Enter } \text{main}()$$

**Specification:**

Implement an extensible, self-registering factory:

1.  Define abstract base class `IPlugin` and factory class
    `PluginFactory`.
2.  Ensure `PluginFactory` uses **Meyer's Singleton**
    (`static PluginFactory& instance()`) to prevent the *Static
    Initialization Order Fiasco*.
3.  Write a helper macro or template `RegisterPlugin<Derived>` that
    registers a type using a string key.
4.  Define two separate plugins (`CompressionPlugin` and `EchoPlugin`)
    in isolated translation units, verifying that they can be
    instantiated by name from `main()` without explicit registration
    calls.

``` cpp
#include <string>
#include <unordered_map>
#include <functional>
#include <memory>
#include <stdexcept>

class IPlugin {
public:
    virtual ~IPlugin() = default;
    virtual void execute() = 0;
};

class PluginFactory {
public:
    using Creator = std::function<std::unique_ptr<IPlugin>()>;
    static PluginFactory& instance() {
        static PluginFactory factory; // Guaranteed thread-safe initialisation
        return factory;
    }
    void register_plugin(const std::string& name, Creator creator) {
        m_creators[name] = std::move(creator);
    }
    std::unique_ptr<IPlugin> create(const std::string& name) const {
        auto it = m_creators.find(name);
        if (it == m_creators.end()) throw std::runtime_error("Unknown plugin: " + name);
        return (it->second)();
    }
private:
    PluginFactory() = default;
    std::unordered_map<std::string, Creator> m_creators;
};
```

**Verification Tasks:**

- **Task 1 (Dynamic Instantiation):** Request `"Compression"` from the
  factory. Verify that a valid `IPlugin` pointer is returned and its
  `execute()` method runs.
- **Task 2 (Unregistered Key Failure):** Request an unregistered string
  `"SpatialAudio"`. Verify that the factory throws a
  `std::runtime_error`.
- **Task 3 (Zero-Touch Extensibility):** Create a new derived class in a
  separate `.cpp` file without including any header files other than
  `IPlugin.hpp`. Recompile and verify that the new type is available in
  the factory.

#### Exercise 13.11: The Adapter Pattern: Modernising Legacy C APIs

**Context and Theory:**

Systems programming often requires integrating legacy C libraries or
operating system system calls into Modern C++ applications. Legacy APIs
typically feature error-prone conventions:

- Raw, unmanaged memory buffers (`char* buf, int max_len`).
- Integer error return codes (`-1` on failure, `0` on success).
- Implicit manual cleanup functions (`legacy_free(ptr)`).

Directly scattering calls to legacy APIs throughout an application
tightly couples the codebase to obsolete idioms and increases the risk
of memory leaks and buffer overflows.

The **Adapter Pattern** converts the interface of an existing class or C
API into an interface that clients expect.

A Modern C++ Adapter encapsulates legacy calls behind an idiomatic
interface:

- Raw output pointers are adapted into value types (`std::string`,
  `std::vector`).
- Error codes are adapted into `std::expected` or `std::optional`.
- Raw handles are wrapped in RAII structures to guarantee cleanup.

**Specification:**

Adapt a legacy C-style sensor API:

1.  Given legacy C function:
    `int c_read_sensor_data(int id, char* out_buffer, int max_len);`
    (returns bytes written, or `-1` on error).
2.  Define a clean modern interface: `ISensorService` declaring
    `virtual std::optional<std::string> read_sensor(int id) = 0;`.
3.  Implement `LegacySensorAdapter` implementing `ISensorService`:
    - Manages buffer allocation internally.
    - Traps negative error codes, returning `std::nullopt`.
    - Automatically formats and bounds-checks the returned string.

``` cpp
#include <string>
#include <optional>
#include <vector>

// Simulated Legacy C API
int c_read_sensor_data(int id, char* out_buffer, int max_len);

class ISensorService {
public:
    virtual ~ISensorService() = default;
    [[nodiscard]] virtual std::optional<std::string> read_sensor(int id) = 0;
};

class LegacySensorAdapter final : public ISensorService {
public:
    [[nodiscard]] std::optional<std::string> read_sensor(int id) override {
        std::vector<char> buffer(256);
        const int bytes_read = c_read_sensor_data(id, buffer.data(), static_cast<int>(buffer.size()));
        if (bytes_read < 0) return std::nullopt;
        return std::string(buffer.data(), static_cast<std::size_t>(bytes_read));
    }
};
```

**Verification Tasks:**

- **Task 1 (Successful Data Adaptation):** Configure the legacy C mock
  to return `"TEMP: 24.5C"`. Call `read_sensor(1)`. Verify it returns a
  `std::optional` containing the correct string.
- **Task 2 (Error Translation):** Configure the legacy C mock to return
  `-1` (hardware error). Call `read_sensor(2)`. Verify that the adapter
  catches the error code and returns `std::nullopt`.
- **Task 3 (Buffer Overflow Prevention):** Configure the legacy C mock
  to produce 500 bytes of output. Verify that the adapter's bounded
  buffer prevents stack smashing or memory corruption.

#### Exercise 13.12: Dynamic Plugin Loading via Abstract Factory & Shared Libraries

**Context and Theory:**

Enterprise architectures, game engines, and IDEs often support
third-party extensibility through **Dynamic Plugins**. A dynamic plugin
is a compiled shared library (`.so` on Linux, `.dylib` on macOS, `.dll`
on Windows) that is loaded into the host process's virtual memory at
runtime using operating system APIs (`dlopen()` / `LoadLibrary()`).

Because C++ does not have a standardized Application Binary Interface
(ABI), loading C++ classes directly across dynamic library boundaries
can cause undefined behavior if the host and plugin were compiled with
different compilers or optimization flags.

To achieve safe dynamic loading, systems combine the **Abstract Factory
Pattern** with a **C-Compatible ABI Interface**:

1.  The shared library exports a single raw C function (`extern "C"`):
    `IPluginFactory* get_plugin_factory();`
2.  The host calls this function via `dlsym()` to retrieve a pointer to
    the abstract factory.
3.  The host then invokes the factory's virtual methods (`create()`) to
    instantiate Modern C++ objects safely within the host's memory
    space.

**Specification:**

Design a dynamic audio filter plugin loader:

1.  Define abstract base class `IAudioFilter` declaring
    `virtual void process(std::span<float> buffer) = 0;`.
2.  Define `IFilterFactory` declaring
    `virtual std::unique_ptr<IAudioFilter> create() = 0;`.
3.  Implement a `PluginLoader` class that encapsulates the dynamic
    loading lifecycle:
    - Takes the path to a shared object.
    - Resolves the entry-point factory symbol using `dlsym()`.
    - Returns a factory instance wrapped in an RAII handle that calls
      `dlclose()` upon destruction.

``` cpp
#include <span>
#include <memory>
#include <string>
#include <stdexcept>

class IAudioFilter {
public:
    virtual ~IAudioFilter() = default;
    virtual void process(std::span<float> samples) = 0;
};

class IFilterFactory {
public:
    virtual ~IFilterFactory() = default;
    virtual std::unique_ptr<IAudioFilter> create() = 0;
};

// C-ABI Export Signature
using GetFactoryFunc = IFilterFactory* (*)();

class PluginHandle {
    void* m_lib_handle{nullptr};
    IFilterFactory* m_factory{nullptr};
public:
    PluginHandle(const std::string& path);
    ~PluginHandle(); // Invokes dlclose(m_lib_handle)
    [[nodiscard]] IFilterFactory* factory() const noexcept { return m_factory; }
};
```

**Verification Tasks:**

- **Task 1 (Simulated Factory Invocation):** Compile a mock plugin
  library that exports a `VolumeFilter` (attenuating samples by $0.5$).
  Load the plugin and instantiate the filter. Pass a buffer of
  `[1.0, -1.0, 0.5]`. Verify that samples are modified to
  `[0.5, -0.5, 0.25]`.
- **Task 2 (Missing Library Exception):** Attempt to load a non-existent
  file path `"missing_filter.so"`. Verify that the loader throws a
  `std::runtime_error` containing diagnostic information.
- **Task 3 (RAII Library Teardown):** Verify that discarding the
  `PluginHandle` unloads the shared object from process memory cleanly
  without leaking dynamic linker resources.